# Publication Figures -- Dogwhistle Benchmarking Audit

Generates every figure actually embedded in the paper (`paper/acl_latex.tex` /
`paper/WOAH26_BDW.pdf`): 13 figures from `generate_figures_final.py`'s logic
(`fig1`-`fig7`, `appA`-`appG`; `appD` is generated but not embedded -- see its
own cell) plus the pairwise annotation-DI figure from
`generate_fig2_annotation_di_pairwise.py`'s logic (Figure 2). Both of those
root-level scripts are archived (`deprecated/generate_figures_final.py`,
`deprecated/generate_fig2_annotation_di_pairwise.py`) -- this notebook is
their sole living replacement; see the "Style Pass Summary" and "Production
promotion" cells below for how it got there.

**Split on 2026-09-10** from a larger notebook that also reproduced two
figure-generation pipelines not used by the paper (`stage5_figures.py`, the
deprecated `auditing/` pipeline) plus a one-time cross-stage consistency
check. That content now lives in
`deprecated/legacy_figures_consolidated.ipynb`, writing under
`deprecated/legacy_outputs/` -- this notebook and `outputs/` now contain only
paper-relevant figures. Output paths were renamed at the same time to match
each figure's number as compiled in the current PDF (`fig01`-`fig16`,
`fig_unused_*` for the two generated-but-not-embedded figures); see
`scratch/figure_swap_tracker.md` for the full old-name/new-name/figure-number
mapping.

**Output paths are preserved exactly as in the source files** so nothing
downstream (the paper's LaTeX includes) breaks. Figures are generated
strictly from the primary ("full", all glossary tiers) analysis; the
tier1+2 robustness-check variant is a separate comparison
(`audit_pipeline/robustness_check.py`), not a figure.

## Setup -- imports and repository paths

`REPO_ROOT` is computed explicitly (rather than relying on notebook cwd) so
figure generation works regardless of where Jupyter's working directory
happens to be.

No sandbox mode: `OUTPUTS_DIR` always resolves to the real `outputs/` tree
the paper's LaTeX includes read from. There is deliberately no debug/dry-run
mode to leave on by accident -- if you need to preview changes without
touching production paths, copy `OUTPUTS_DIR` to a temp directory yourself
before running.

This notebook has no dependency on `audit_pipeline.config` /
`audit_pipeline.helpers` -- every figure below reads its inputs as plain TSV
paths under `OUTPUTS_DIR` and hardcodes the 0.80 four-fifths-rule threshold
directly (verified: nothing in `generate_figures_final.py` or
`generate_fig2_annotation_di_pairwise.py`'s original logic referenced
`PipelineVariant` or the config module's `DI_THRESHOLD`/`N_MIN` either).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "audit_pipeline").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Could not locate repo root (no audit_pipeline/ found above cwd).")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUTPUTS_DIR = REPO_ROOT / "outputs"
GFF_DATA_DIR = str(OUTPUTS_DIR) + "/"        # generate_figures_final.py's data_dir param

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from pandas.errors import EmptyDataError

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"OUTPUTS_DIR = {OUTPUTS_DIR}")

## `generate_figures_final.py` -- shared utilities

Copied verbatim: the `STYLE` dict, `apply_style()`, `_read()`, `_need()`,
`_save()`, `_panel_label()`, and `_declutter_labels()` (used by
`appF_cross_level_deltas` and, as of the fig13 label-collision fix below,
`appG_elsherief_pairwise_di_delta`). These are the ACL/EMNLP camera-ready styling
helpers -- distinct from, and **not** merged with, the `_s5`-suffixed
helpers above (different DPI: 300 vs 220; different format: PDF vs PNG;
different `tight_layout`/`constrained_layout` handling). Every figure
function below calls these under their original (unsuffixed) names, exactly
as in the source file.

As with `_save_s5` above, one line was added to `_save` that is not present
in the original `generate_figures_final.py`: a `plt.show()` call right
before `plt.close(fig)`, so each figure renders inline in this notebook's
cell output. Marked with an `# added:` comment; no figure content changed.


In [ ]:
STYLE = {
    "title": None,
    "axis_label_fontsize": 10,
    "tick_label_fontsize": 8,
    "legend_fontsize": 8,
    "annotation_fontsize": 8,
    "panel_label_fontsize": 10,
    "colors": {
        "L2": "#E69F00",
        "L3": "#56B4E9",
        "L4": "#CC79A7",
        "correct": "#009E73",
        "failure": "#D55E00",
        "case_a": "#009E73",
        "case_b": "#D55E00",
        "presence": "#0072B2",
        "type_cov": "#D55E00",  # was #56B4E9 (sky-blue, same hex as L3 label -- indistinguishable from presence blue at a glance): switched to vermilion for a clearly distinct pair, per author request
        "pass": "#009E73",  # was #0072B2 (blue): standardized to teal to match correct/case_a, per author request (fig10 pass/fail bars)
        "fail": "#D55E00",
        "delta_pos": "#009E73",
        "delta_neg": "#D55E00",
        "neutral": "#999999",
    },
    "dpi": 300,
    "format": "pdf",
    "bbox_inches": "tight",
    "pad_inches": 0.05,
    "columnwidth_in": 3.35,
    "textwidth_in": 6.97,
}

def apply_style():
    """Apply ACL/EMNLP rcParams at the start of every figure function."""
    mpl.rcParams.update(
        {
            "font.size": STYLE["tick_label_fontsize"],
            "axes.labelsize": STYLE["axis_label_fontsize"],
            "xtick.labelsize": STYLE["tick_label_fontsize"],
            "ytick.labelsize": STYLE["tick_label_fontsize"],
            "legend.fontsize": STYLE["legend_fontsize"],
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.dpi": STYLE["dpi"],
            # 2026-07-03: explicitly reset -- stage5_figures.py's utilities cell
            # (run earlier in this notebook, same kernel) sets these globally via
            # its own mpl.rcParams.update() and apply_style() never reset them
            # back, so every gff figure was silently inheriting stage5's grid
            # lines + off-white background. Visible as three vertical white grid
            # lines through fig1_coverage_heatmap's three columns.
            "axes.grid": False,
            "axes.facecolor": "white",
        }
    )

def _read(path: str | Path) -> pd.DataFrame:
    """Read a TSV file; return empty DataFrame with a warning if missing."""
    p = Path(path)
    if not p.exists():
        print(f"WARNING: Missing file {p}")
        return pd.DataFrame()
    return pd.read_csv(p, sep="\t", low_memory=False)

def _need(paths: list[str | Path], fig_name: str) -> bool:
    """Return True iff every path exists; print warnings for any missing."""
    missing = [p for p in paths if not Path(p).exists()]
    for m in missing:
        print(f"WARNING: Skipping {fig_name}: missing {m}")
    return len(missing) == 0

def _save(fig: plt.Figure, path: str | Path) -> None:
    """Save figure as PDF with ACL standard settings, then close.

    Calls tight_layout only for figures that do NOT use constrained_layout
    (constrained_layout handles its own spacing; calling tight_layout on top
    of it overrides it and breaks legend placement).
    """
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if not fig.get_constrained_layout():
        # fig1 uses manual add_axes; tight_layout skips incompatible axes silently
        try:
            fig.tight_layout(pad=0.4)
        except Exception:
            pass
    fig.savefig(str(p), format="pdf", bbox_inches="tight", pad_inches=0.08)
    plt.show()  # added: display the figure inline in the notebook output before closing it
    plt.close(fig)
    print(f"Generated: {p}")

def _panel_label(ax: plt.Axes, label: str, color: str) -> None:
    """Place a level label (L2 / L3 / L4) just above the upper-left corner."""
    ax.text(
        0.03,
        1.01,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=STYLE["panel_label_fontsize"],
        fontweight="bold",
        color=color,
        clip_on=False,
    )

def _declutter_labels(
    ax: plt.Axes,
    fig: plt.Figure,
    end_points: list[tuple[str, float, float, str]],
    fontsize: float,
    x_pad_px: float = 6.0,
) -> None:
    """Place a direct end-of-line label per group, greedily decluttered in
    pixel space so labels closer than ~one line-height collapse into a
    vertically stacked, non-overlapping column.

    end_points: list of (label_text, x_final_data, y_final_data, color).
    Decluttering must happen in *display* pixels, not data coordinates,
    because the data only spans [0, 1] while actual visual separation is
    governed by rendered font size at the figure's draw DPI.
    """
    if not end_points:
        return

    fig.canvas.draw()
    dpi = fig.dpi
    # One line-height at the render DPI, plus a little leading, is the
    # minimum gap that avoids glyphs from adjacent labels touching.
    min_gap_px = fontsize * (dpi / 72.0) * 1.7  # was 1.25 -- more breathing room (2026-07-03 style pass, appF label crowding)
    edge_margin_px = min_gap_px * 0.4

    bbox = ax.get_window_extent()
    top_bound = bbox.y1 - edge_margin_px
    bottom_bound = bbox.y0 + edge_margin_px

    items = []
    for label_text, x_final, y_final, color in end_points:
        px, py = ax.transData.transform((x_final, y_final))
        items.append([label_text, x_final, y_final, color, px, py])

    # Top to bottom on screen = descending pixel y.
    items.sort(key=lambda it: -it[5])

    # Pass 1 (top-down): push each label below the one above it, clamped to
    # the axes' top edge.
    resolved_py = []
    prev_py = None
    for it in items:
        py = min(it[5], top_bound)
        cur = py if prev_py is None else min(py, prev_py - min_gap_px)
        resolved_py.append(cur)
        prev_py = cur

    # Pass 2 (bottom-up): if the stack ran past the bottom edge, pull labels
    # back up so the whole column fits within the axes.
    if resolved_py[-1] < bottom_bound:
        resolved_py[-1] = bottom_bound
        for i in range(len(resolved_py) - 2, -1, -1):
            resolved_py[i] = max(resolved_py[i], resolved_py[i + 1] + min_gap_px)

    inv = ax.transData.inverted()
    for it, py_resolved in zip(items, resolved_py):
        label_text, x_final, y_final, color, px, _py = it
        x_label, y_label = inv.transform((px + x_pad_px, py_resolved))
        ax.annotate(
            label_text,
            xy=(x_final, y_final),
            xycoords="data",
            xytext=(x_label, y_label),
            textcoords="data",
            ha="left",
            va="center",
            fontsize=fontsize,
            color=color,
            clip_on=False,
            zorder=4,
        )


## "Camera-ready" figures (`generate_figures_final.py`)

13 standalone figure functions (`fig1`-`fig7`, `appA`-`appG`), each already
documenting its own source file(s) in its docstring -- reproduced below
verbatim. This script is **not** `PipelineVariant`-aware: every function
takes a plain `data_dir` string and always reads/writes under it, so it can
only ever target the primary ("full") analysis outputs, never the tier1+2
robustness check. That is a genuine capability gap versus `stage5_figures.py`,
noted here rather than silently patched.


### `fig1_coverage_heatmap`

**Output file:** `outputs/figures_final/fig05_coverage_heatmap.pdf`

**Source TSV(s):** `outputs/stage1/s1_coverage_by_level_target.tsv`, `outputs/unioned_data/06_glossary_label_reference.tsv`

**Pipeline stage:** Stage 1 (`stage1/s1_coverage_by_level_target.tsv`) + upstream preprocessing output (`unioned_data/06_glossary_label_reference.tsv`, produced before Stage 1 by the `data_preprocessing/` notebooks)

**Docstring (verbatim from source):**

```
Heatmap of dogwhistle presence_rate for each (reporting group × coding level).

Source files:
- stage1/s1_coverage_by_level_target.tsv  : presence_rate, type_coverage per cell
- unioned_data/06_glossary_label_reference.tsv : confirms entry existence per cell

Visual encoding:
- Sequential green (light→dark) = presence_rate 0→1
- Gray cell = no glossary entries at that coding level for that group
- Red border + asterisk in cell text = type_coverage < 1.0
- Thin black separator lines between reporting dimension groups
- Right-side italic dimension label per group
```


In [ ]:
def fig1_coverage_heatmap(data_dir: str = "outputs/") -> None:
    """
    Heatmap of dogwhistle presence_rate for each (reporting group × coding level).

    Source files:
    - stage1/s1_coverage_by_level_target.tsv  : presence_rate, type_coverage per cell
    - unioned_data/06_glossary_label_reference.tsv : confirms entry existence per cell

    Visual encoding:
    - Sequential green (light→dark) = presence_rate 0→1
    - Gray cell = no glossary entries at that coding level for that group
    - Red border + asterisk in cell text = type_coverage < 1.0
    - Thin black separator lines between reporting dimension groups
    - Right-side italic dimension label per group
    """
    apply_style()
    base = Path(data_dir)
    s1_p = base / "stage1/s1_coverage_by_level_target.tsv"
    gl_p = base / "unioned_data/06_glossary_label_reference.tsv"
    if not _need([s1_p, gl_p], "fig1"):
        return

    s1 = _read(s1_p)
    if s1.empty:
        return

    # Map taxonomy_level → display dimension; filter to the 7 reported dimensions
    DIM_MAP = {
        "race": "race",
        "religion": "religion",
        "sexuality": "lgbtq",
        "gender": "gender",
        "origin": "origin",
        "politics": "politics",
        "disability": "disability",
    }
    DIM_ORDER = [
        "race",
        "religion",
        "lgbtq",
        "gender",
        "origin",
        "politics",
        "disability",
    ]
    LEVELS = ["L2", "L3", "L4"]
    COL_LABELS = ["L2", "L3", "L4"]

    s1 = s1[s1["taxonomy_level"].isin(DIM_MAP)].copy()
    # Explicit target exclusions: self-referential and pan-group categories excluded
    # from pipeline reporting (report_include = False).  The taxonomy_level filter
    # handles most cases, but defence-in-depth avoids silent leakage if taxonomy
    # assignment changes in a future pipeline run.
    _EXCLUDE = {"referential_white_supremacist", "minority"}
    s1 = s1[~s1["target"].isin(_EXCLUDE)].copy()
    s1["dimension"] = s1["taxonomy_level"].map(DIM_MAP)

    # Build ordered row list: DIM_ORDER, then alphabetical within each dimension
    rows: list[tuple[str, str]] = []  # (target, dimension)
    dim_ranges: dict[str, tuple[int, int]] = {}
    for dim in DIM_ORDER:
        targets = sorted(s1[s1["dimension"] == dim]["target"].unique())
        start = len(rows)
        rows.extend((t, dim) for t in targets)
        dim_ranges[dim] = (start, len(rows))

    n_rows, n_cols = len(rows), len(LEVELS)

    # Build grids (indexed [row, col])
    pr_grid = np.full((n_rows, n_cols), np.nan)
    tc_grid = np.full((n_rows, n_cols), np.nan)
    has_entry = np.zeros((n_rows, n_cols), dtype=bool)

    for i, (target, _) in enumerate(rows):
        for j, level in enumerate(LEVELS):
            mask = (s1["target"] == target) & (s1["coding_level"] == level)
            sub = s1[mask]
            if not sub.empty:
                has_entry[i, j] = True
                pr_grid[i, j] = float(sub["presence_rate"].iloc[0])
                tc_grid[i, j] = float(sub["type_coverage"].iloc[0])

    # Custom green colormap: clipped so 0% maps to light (not white) green
    base_cmap = mpl.colormaps.get_cmap("Greens")
    greens = LinearSegmentedColormap.from_list(
        "greens_clipped",
        [
            base_cmap(0.15),
            base_cmap(0.38),
            base_cmap(0.62),
            base_cmap(0.85),
            base_cmap(1.0),
        ],
    )
    greens.set_bad(color="#cccccc")  # Gray for missing cells

    pr_masked = np.ma.masked_where(~has_entry, pr_grid)

    # Figure layout: main heatmap + colorbar below
    cell_h = 0.36  # inches per row
    cbar_h = 0.55  # inches for colorbar + legend area
    main_h = n_rows * cell_h
    fig_h = main_h + cbar_h + 0.5
    fig_w = fig_h * 0.85  # was STYLE["textwidth_in"] * 0.67 (fixed, independent of content height, giving an extreme ~0.35:1 final aspect); tied to fig_h for a 2:3 (width:height) FINAL aspect per author request. Empirically calibrated: bbox_inches="tight" trims unused right-side margin at this nominal figsize, so a plain fig_h*2/3 nominal width under-shoots the target 2:3 in the saved PDF -- 0.85 hits ~2:3 after that trim. .tex embed width (0.67\textwidth) is unchanged, so the wider source makes the final printed height shrink to a reasonable single-page size instead of the previous ~13in.

    fig = plt.figure(figsize=(fig_w, fig_h))

    # Reserve proportional space: [left, bottom, width, height] in figure fraction
    left_frac = 0.22
    right_pad = 0.18  # space for dimension labels
    main_width = 1.0 - left_frac - right_pad
    main_bottom = cbar_h / fig_h + 0.04
    main_height = main_h / fig_h

    ax = fig.add_axes([left_frac, main_bottom, main_width, main_height])
    ax.set_title("")

    # Draw heatmap
    im = ax.imshow(
        pr_masked,
        cmap=greens,
        vmin=0,
        vmax=1,
        aspect="auto",
        extent=[-0.5, n_cols - 0.5, n_rows - 0.5, -0.5],  # x: cols, y: rows (0 at top)
    )

    # Cell annotations and red borders
    for i in range(n_rows):
        for j in range(n_cols):
            if not has_entry[i, j]:
                continue
            pr = pr_grid[i, j]
            tc = tc_grid[i, j]
            label = f"{int(round(pr * 100))}%{'*' if tc < 1.0 else ''}"
            text_color = "white" if pr > 0.55 else "black"
            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=STYLE["annotation_fontsize"],
                color=text_color,
            )
            if tc < 1.0:
                rect = mpatches.Rectangle(
                    (j - 0.5, i - 0.5),
                    1.0,
                    1.0,
                    linewidth=2,
                    edgecolor="red",
                    facecolor="none",
                    clip_on=False,
                )
                ax.add_patch(rect)

    # Dimension separator lines
    for dim in DIM_ORDER:
        _, end_idx = dim_ranges[dim]
        if end_idx < n_rows:
            ax.axhline(end_idx - 0.5, color="black", linewidth=0.9, clip_on=False)

    # Right-side dimension labels (in axes transData, extended x range)
    for dim in DIM_ORDER:
        s_idx, e_idx = dim_ranges[dim]
        mid_y = (s_idx + e_idx - 1) / 2
        ax.text(
            n_cols - 0.5 + 0.25,
            mid_y,
            dim,
            ha="left",
            va="center",
            fontsize=STYLE["tick_label_fontsize"],
            fontstyle="italic",
            transform=ax.transData,
            clip_on=False,
        )

    # Column headers (on top)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(COL_LABELS, fontsize=STYLE["tick_label_fontsize"])
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")

    # Y-axis: target names
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels([t for t, _ in rows], fontsize=STYLE["tick_label_fontsize"])

    ax.set_xlim(-0.5, n_cols - 0.5 + 1.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.tick_params(left=False, top=False, bottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Layout constants for the bottom strip (in inches, then converted to fractions)
    # gap_in must accommodate colorbar tick labels + "Presence rate" label beneath the bar
    legend_h_in = 0.22  # single-row legend height
    legend_bot_in = 0.04  # clearance below legend to figure bottom
    gap_in = 0.7  # was 0.38 -- more clearance: colorbar tick labels collided with "Presence rate" caption after the 0.67x width resize
    cbar_h_in = 0.20  # colorbar bar height
    gap2_in = 0.12  # gap between colorbar top and main heatmap bottom
    bottom_strip = legend_bot_in + legend_h_in + gap_in + cbar_h_in + gap2_in

    legend_bot = legend_bot_in / fig_h
    cbar_bottom = (legend_bot_in + legend_h_in + gap_in) / fig_h
    cbar_ht = cbar_h_in / fig_h

    # Recalculate main heatmap bottom to clear the bottom strip
    main_bottom = bottom_strip / fig_h
    ax.set_position([left_frac, main_bottom, main_width, main_height])

    # Horizontal colorbar
    cbar_left = left_frac
    cbar_width = main_width * 0.78
    cax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_ht])
    cb = mpl.colorbar.ColorbarBase(
        cax, cmap=greens, norm=Normalize(vmin=0, vmax=1), orientation="horizontal"
    )
    cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cb.set_ticklabels(
        ["0%", "25%", "50%", "75%", "100%"], fontsize=STYLE["tick_label_fontsize"]
    )
    cb.set_label("Presence rate", fontsize=STYLE["axis_label_fontsize"])

    # Legend in a single horizontal row below the colorbar (no overlap)
    legend_handles = [
        mpatches.Patch(
            facecolor="#cccccc",
            edgecolor="gray",
            linewidth=0.5,
            label="No glossary entries at this level",
        ),
        mpatches.Patch(
            facecolor=greens(0.0),
            edgecolor="gray",
            linewidth=0.5,
            label="0% presence (entries exist, none found)",
        ),
        mpatches.Patch(
            facecolor="white",
            edgecolor="red",
            linewidth=2,
            label="Type coverage < 100% (* in label)",
        ),
    ]
    # Anchor to the left figure edge so the legend sits cleanly below the colorbar
    fig.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.01, legend_bot),
        ncol=1,
        fontsize=7,
        frameon=False,
        handlelength=1.2,
        handletextpad=0.4,
    )

    _save(fig, base / "figures_final/fig05_coverage_heatmap.pdf")


In [ ]:
fig1_coverage_heatmap(data_dir=GFF_DATA_DIR)


### `annotation_rates_by_level`

**Renamed** from `fig2_annotation_rates_by_level` during the 2026-07-03 style
pass. The `fig2_` prefix collided with `generate_fig2_annotation_di_pairwise.py`'s
`fig02_annotation_di_by_pair_level.pdf` -- the paper's *actual* Figure 2. Confirmed
against `acl_latex (5).tex` (the current draft, the only one of 14 candidate
`.tex` exports found that references `fig02_annotation_di_by_pair_level.pdf` at
all) that this figure is the **first** `\begin{figure}` environment in the main
body (`\label{fig:annotation_rates}`), i.e. **Figure 1** -- not "Figure 8" as
originally believed when this rename was scoped. Given the paper's figure order
has already shifted at least twice during this project, the output filename was
made **position-independent** (no number at all) per author direction, so this
class of staleness cannot recur.

**Output file:** `outputs/figures_final/fig01_annotation_rates_by_level.pdf`
(was `fig2_fig01_annotation_rates_by_level.pdf`)

**Source TSV(s):** `outputs/stage2/s2_annotation_by_level_target.tsv`

**Pipeline stage:** Stage 2 (`stage2/s2_annotation_by_level_target.tsv`)

**`.tex` embed width:** `\columnwidth` (single-column, ACL two-column layout)
-- relevant for the Step 4 sizing pass below, since the function currently
sizes itself at `STYLE["textwidth_in"]` (full double-column width).

**Docstring (verbatim from source):**

```
Three-panel horizontal bar chart: correct vs. failure rates per group × level.

Source files:
- stage2/s2_annotation_by_level_target.tsv

Visual encoding:
- Teal bars = correct_labeling_rate (Case A / (A+B))
- Vermillion bars = failure_rate (Case B / (A+B))
- Only groups with stable match counts (stable_n == True) shown
- Groups sorted by failure_rate descending within each panel
- Panel labels in level colors; legend in L2 panel only
```


In [ ]:
def annotation_rates_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar chart: correct vs. failure rates per group × level.

    Source files:
    - stage2/s2_annotation_by_level_target.tsv

    Visual encoding:
    - Teal bars = correct_labeling_rate (Case A / (A+B))
    - Vermillion bars = failure_rate (Case B / (A+B))
    - Only groups with stable match counts (stable_n == True) shown
    - Groups sorted by failure_rate descending within each panel
    - Panel labels in level colors; legend in L2 panel only
    """
    apply_style()
    base = Path(data_dir)
    s2_p = base / "stage2/s2_annotation_by_level_target.tsv"
    if not _need([s2_p], "annotation_rates_by_level"):
        return

    s2 = _read(s2_p)
    if s2.empty:
        return

    s2 = s2[s2["stable_n"].astype(bool)].copy()
    s2 = s2[~s2["is_self_referential"].astype(bool)].copy()
    # Explicit exclusion: referential_white_supremacist has is_self_referential=False
    # at L3 in the pipeline data, so the flag alone is insufficient.
    # minority is a pan-group category excluded from per-group reporting.
    _EXCLUDE = {"referential_white_supremacist", "minority"}
    s2 = s2[~s2["target"].isin(_EXCLUDE)].copy()

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(6.0, 6.0),  # was (columnwidth_in, 6.0)=(3.35,6.0): widened for 1:1 aspect per author request
        sharey=False,
        constrained_layout=True,
    )

    # Max category count across panels: gives every bar the same physical
    # thickness regardless of panel, so a sparse panel (e.g. L4, with far
    # fewer stable groups than L2/L3) shows genuine blank space instead of
    # its few bars stretching to fill the whole axis height.
    max_n = max(1, max((s2["coding_level"] == lvl).sum() for lvl in LEVELS))

    for col_i, (ax, level) in enumerate(zip(axes, LEVELS)):
        ax.set_title("")
        sub = s2[s2["coding_level"] == level].sort_values(
            "failure_rate", ascending=True
        )
        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["target"].tolist()
        n = len(groups)
        ROW_SPACING = 0.6  # tighter row spacing (was 1.0 implicit via np.arange), matches fig3/fig6's row density
        y = np.arange(n) * ROW_SPACING

        # Stacked (was two dodged bars per category, offset +-BAR_H/2) --
        # correct_labeling_rate + failure_rate always sum to 1, so one bar
        # per category with the two rates concatenated communicates the same
        # information more compactly. Matches appA_coverage_presence_vs_type_by_level's
        # stacking convention, per author request.
        correct = sub["correct_labeling_rate"].values
        failure = sub["failure_rate"].values
        ax.barh(y, correct, color=STYLE["colors"]["correct"], height=0.4, label="Correct")  # was default height=0.8: bar height roughly matching text label height per author request
        ax.barh(y, failure, left=correct, color=STYLE["colors"]["failure"], height=0.4, label="Failure")

        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=8)  # bumped from 7: bigger text per author request
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # consistent bar thickness across panels, scaled by ROW_SPACING (was auto-scaled per panel, making L4's few bars look much thicker/wider than L2/L3's)
        ax.set_xlim(0, 1)
        ax.set_xticks([0, 0.5, 1.0])  # restored per author request: panels wide enough now that 3 ticks no longer collide
        ax.tick_params(axis="x", labelsize=7)  # bumped back toward normal: panels wide enough now
        ax.set_xlabel("Rate", fontsize=7)  # was STYLE["axis_label_fontsize"]=10; narrow panel
        ax.axvline(0, color="black", linewidth=0.5)

        _panel_label(ax, level, STYLE["colors"][level])
        ax.spines["left"].set_visible(False)
        ax.tick_params(left=False)

    legend_handles = [
        mpatches.Patch(facecolor=STYLE["colors"]["correct"], label="Correct"),
        mpatches.Patch(facecolor=STYLE["colors"]["failure"], label="Failure"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        ncol=2,
        fontsize=STYLE["legend_fontsize"],
        frameon=False,
    )

    _save(fig, base / "figures_final/fig01_annotation_rates_by_level.pdf")


In [ ]:
annotation_rates_by_level(data_dir=GFF_DATA_DIR)


### `fig3_case_ab_counts_by_level`

**Output file:** `outputs/figures_final/fig09_case_ab_counts_by_level.pdf`

**Source TSV(s):** `outputs/stage2/s2_annotation_by_level_target.tsv`

**Pipeline stage:** Stage 2 (`stage2/s2_annotation_by_level_target.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal stacked bar: total matches (Case A + Case B) per group × level.

Source files:
- stage2/s2_annotation_by_level_target.tsv

Visual encoding:
- Green segment = Case A (hateful context, correct identification)
- Vermillion segment = Case B (non-hateful context, false positive)
- Top 10 groups per panel by total match count
- If one bar dominates (>3× the 90th percentile of top-10), x-axis is
  truncated and the bar is annotated with its actual count
```


In [ ]:
def fig3_case_ab_counts_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal stacked bar: total matches (Case A + Case B) per group × level.

    Source files:
    - stage2/s2_annotation_by_level_target.tsv

    Visual encoding:
    - Green segment = Case A (hateful context, correct identification)
    - Vermillion segment = Case B (non-hateful context, false positive)
    - Top 10 groups per panel by total match count
    - If one bar dominates (>3× the 90th percentile of top-10), x-axis is
      truncated and the bar is annotated with its actual count
    """
    apply_style()
    base = Path(data_dir)
    s2_p = base / "stage2/s2_annotation_by_level_target.tsv"
    if not _need([s2_p], "fig3"):
        return

    s2 = _read(s2_p)
    if s2.empty:
        return

    s2 = s2[~s2["is_self_referential"].astype(bool)].copy()
    _EXCLUDE = {"referential_white_supremacist", "minority"}
    s2 = s2[~s2["target"].isin(_EXCLUDE)].copy()

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(8.0, 4.0), constrained_layout=True  # was (columnwidth_in, 4.0)=(3.35,4.0): widened to 2:1 aspect per author request (also fixes rightmost panel's "Match count" label getting cut off at the narrower width)
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = (
            s2[s2["coding_level"] == level]
            .dropna(subset=["total_matches"])
            .nlargest(10, "total_matches")
            .sort_values("total_matches", ascending=True)
        )

        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["target"].tolist()
        n = len(groups)
        ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars
        y = np.arange(n) * ROW_SPACING
        totals = sub["total_matches"].values
        ca = sub["case_a_present_hateful"].values
        cb = sub["case_b_present_nonhateful"].values

        # Determine x axis limit; truncate if one bar dominates
        pct90 = np.percentile(totals, 90) if len(totals) > 1 else totals[0]
        x_max = totals.max() * 1.05
        truncate = False
        if totals.max() > 3 * pct90 and len(totals) > 2:
            sorted_t = np.sort(totals)
            x_max = sorted_t[-2] * 1.5
            truncate = True

        ax.barh(y, ca, color=STYLE["colors"]["case_a"], height=0.4, label="Case A (hateful)")  # was 0.6 (before that, default 0.8): further thinned per author request
        ax.barh(
            y,
            cb,
            left=ca,
            height=0.4,
            color=STYLE["colors"]["case_b"],
            label="Case B (non-hateful)",
        )

        # Annotate truncated bars: place text just outside the clipped bar end
        if truncate:
            for yi, tot in zip(y, totals):  # was enumerate(totals) giving raw row index -- misaligned once y became ROW_SPACING-scaled
                if tot > x_max:
                    ax.text(
                        x_max * 1.01,
                        yi,
                        f"{tot:,}",
                        ha="left",
                        va="center",
                        fontsize=STYLE["annotation_fontsize"],
                        color="black",
                        clip_on=False,
                    )

        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=8)  # bumped from 6.5: panels much wider now at 2:1 aspect, room for larger text
        ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)
        ax.set_xlim(0, x_max)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6, rotation removed: panels wide enough now
        ax.set_xlabel("Match count", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panel wide enough now, and no longer at risk of being clipped at the figure edge

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    # Legend below all panels
    legend_handles = [
        mpatches.Patch(facecolor=STYLE["colors"]["case_a"], label="Case A (hateful)"),
        mpatches.Patch(
            facecolor=STYLE["colors"]["case_b"], label="Case B (non-hateful)"
        ),
    ]
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        ncol=2,
        fontsize=STYLE["legend_fontsize"],
        frameon=False,
    )

    _save(fig, base / "figures_final/fig09_case_ab_counts_by_level.pdf")


In [ ]:
fig3_case_ab_counts_by_level(data_dir=GFF_DATA_DIR)


### `fig4_di_histograms_by_level`

**Output file:** `outputs/figures_final/fig03_di_histograms_by_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel overlapping histograms of pairwise DI ratios (presence and type-coverage).

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- Blue histogram = presence_rate_di_ratio (alpha=0.7)
- Sky-blue histogram = type_coverage_di_ratio (alpha=0.7)
- Dashed red vertical line at DI = 0.80 (4/5 rule threshold)
- Only stable pairs (unstable_small_n == False) included
```


In [ ]:
def fig4_di_histograms_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel overlapping histograms of pairwise DI ratios (presence and type-coverage).

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - Blue histogram = presence_rate_di_ratio (alpha=0.7)
    - Sky-blue histogram = type_coverage_di_ratio (alpha=0.7)
    - Dashed red vertical line at DI = 0.80 (4/5 rule threshold)
    - Only stable pairs (unstable_small_n == False) included
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "fig4"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].copy()

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(STYLE["textwidth_in"], 2), constrained_layout=True  # height 3.2->2.7: shorter y-axis / wider aspect per author request
    )

    # DI ratios are bounded [0, 1]; extend slightly to 1.05 to catch floating-point edge
    # values and avoid a spurious spike at the histogram's right edge.
    BINS = np.linspace(0, 1.05, 22)

    for col_i, (ax, level) in enumerate(zip(axes, LEVELS)):
        ax.set_title("")
        sub = df[df["coding_level"] == level]
        pr_di = sub["presence_rate_di_ratio"].dropna()
        tc_di = sub["type_coverage_di_ratio"].dropna()

        ax.hist(
            pr_di,
            bins=BINS,
            color=STYLE["colors"]["presence"],
            alpha=0.7,
            label="Presence DI",
        )
        ax.hist(
            tc_di,
            bins=BINS,
            color=STYLE["colors"]["type_cov"],
            alpha=0.7,
            label="Type-cov. DI",
        )

        ax.axvline(0.80, color="red", linewidth=1.2, linestyle="--")

        ax.set_xlim(0, 1.05)
        ax.set_xlabel("DI ratio", fontsize=STYLE["axis_label_fontsize"])
        ax.set_ylabel(
            "Count" if col_i == 0 else "", fontsize=STYLE["axis_label_fontsize"]
        )
        # Force integer y-axis ticks: with few stable pairs (L3 has 4, L4 has 1),
        # auto-scaling produces fractional counts like 0.25, 0.50 which are meaningless.
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

        _panel_label(ax, level, STYLE["colors"][level])

        if col_i == 0:
            ax.legend(fontsize=STYLE["legend_fontsize"], frameon=False)

    _save(fig, base / "figures_final/fig03_di_histograms_by_level.pdf")


In [ ]:
fig4_di_histograms_by_level(data_dir=GFF_DATA_DIR)


### `fig5_worst_di_by_pair_level`

**Output file:** `outputs/figures_final/fig10_worst_di_by_pair_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal bar chart: worst (minimum) DI ratio per stable group pair × level.

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- Blue bar (pass color) = DI >= 0.80 (passes 4/5 rule)
- Red bar (fail color) = DI < 0.80 (fails 4/5 rule)
- Dashed red vertical line at DI = 0.80
- Only stable pairs (unstable_small_n == False) shown
- Sorted ascending (worst DI at top)
```

**Note:** A **third** distinct source for a "worst pairwise DI" figure (Stage 4b), alongside Stage 3 (`stage5_figures.level_stratified_figures`) and Stage 4a (`stage5_figures.group_collapsed_figures`) above. This is the file `generate_fig2_annotation_di_pairwise.py`'s own docstring calls "Figure 9" while its output filename says `fig2_...` -- another naming inconsistency, surfaced not fixed.


In [ ]:
def fig5_worst_di_by_pair_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar chart: worst (minimum) DI ratio per stable group pair × level.

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - Blue bar (pass color) = DI >= 0.80 (passes 4/5 rule)
    - Red bar (fail color) = DI < 0.80 (fails 4/5 rule)
    - Dashed red vertical line at DI = 0.80
    - Only stable pairs (unstable_small_n == False) shown
    - Sorted ascending (worst DI at top)
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "fig5"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].dropna(subset=["worst_di_ratio"])

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(6.4, 1.4), constrained_layout=True  # was (6.4, 1.8): shrunk further to match tighter ROW_SPACING below, keeping bar thickness (not just whitespace) constant
    )

    # Max bar count across panels: gives every bar the same physical thickness
    # regardless of panel, so a sparse panel (e.g. L4) reads as "mostly empty"
    # rather than a single bar stretched to fill the whole axis. Matches the
    # technique already used in generate_fig2_annotation_di_pairwise.py.
    max_n = max(1, max((df["coding_level"] == lvl).sum() for lvl in LEVELS))

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = df[df["coding_level"] == level].sort_values(
            "worst_di_ratio", ascending=True
        )

        if sub.empty:
            ax.axis("off")
            continue

        pairs = [f"{a} vs {b}" for a, b in zip(sub["target_a"], sub["target_b"])]
        di = sub["worst_di_ratio"].values
        n = len(pairs)
        ROW_SPACING = 0.3  # was 0.45 (before that, 0.6): gap between bars tightened per author request
        y = np.arange(n) * ROW_SPACING

        colors = [
            STYLE["colors"]["pass"] if v >= 0.80 else STYLE["colors"]["fail"]
            for v in di
        ]

        ax.barh(y, di, color=colors, height=0.22)  # was 0.3 (before that, 0.4, 0.6, default 0.8): further thinned per author request
        ax.axvline(0.80, color="red", linewidth=1.2, linestyle="--")

        ax.set_yticks(y)
        ax.set_yticklabels(pairs, fontsize=6)  # bumped from 5: panels wider now at 2:1 aspect
        ax.set_xlim(0, 1.1)
        ax.set_xticks([0, 0.5, 1.0])  # added 0.5 marker per author request; panels wide enough now
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6: panels wider now
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # consistent bar thickness across panels, scaled by ROW_SPACING
        ax.set_xlabel("Worst DI ratio", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panel wide enough now, and no longer at risk of being clipped at the figure edge

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/fig10_worst_di_by_pair_level.pdf")


In [ ]:
fig5_worst_di_by_pair_level(data_dir=GFF_DATA_DIR)


### `fig6_elsherief_annotation_delta`

**Output file:** `outputs/figures_final/fig04a_elsherief_annotation_delta.pdf`

**Source TSV(s):** `outputs/stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`

**Pipeline stage:** Stage 4c (`stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`)

**Docstring (verbatim from source):**

```
Horizontal bar chart: Δ correct labeling rate (union − ElSherief) per reporting group.

Source files:
- stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv

Visual encoding:
- Green bar = Δ > 0 (union improves on ElSherief alone)
- Red bar = Δ < 0 (ElSherief alone outperforms union)
- Sorted ascending (most negative delta at top)

Data treatment: the source file has one row per (coding_level, report_level, report_target).
We aggregate across coding levels by summing the underlying case counts (A and B) for both
the union and ElSherief corpora, then recompute rates from the pooled totals — rates must
not be averaged directly because denominators differ across levels.  Groups with fewer than
20 pooled union matches or with zero ElSherief matches are excluded.
```


In [ ]:
def fig6_elsherief_annotation_delta(data_dir: str = "outputs/") -> None:
    """
    Horizontal bar chart: Δ correct labeling rate (union − ElSherief) per reporting group.

    Source files:
    - stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv

    Visual encoding:
    - Green bar = Δ > 0 (union improves on ElSherief alone)
    - Red bar = Δ < 0 (ElSherief alone outperforms union)
    - Sorted ascending (most negative delta at top)

    Data treatment: the source file has one row per (coding_level, report_level, report_target).
    We aggregate across coding levels by summing the underlying case counts (A and B) for both
    the union and ElSherief corpora, then recompute rates from the pooled totals — rates must
    not be averaged directly because denominators differ across levels.  Groups with fewer than
    20 pooled union matches or with zero ElSherief matches are excluded.
    """
    apply_style()
    base = Path(data_dir)
    s4c_p = base / "stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv"
    if not _need([s4c_p], "fig6"):
        return

    df = _read(s4c_p)
    if df.empty:
        return
    required = {
        "report_level",
        "report_target",
        "case_a_present_hateful_union",
        "case_b_present_nonhateful_union",
        "case_a_present_hateful_elsherief",
        "case_b_present_nonhateful_elsherief",
    }
    missing_cols = required - set(df.columns)
    if missing_cols:
        print(f"WARNING: fig6: skipping — missing columns {missing_cols}")
        return

    # Pool across coding levels: sum case counts, then recompute rates
    agg = df.groupby(["report_level", "report_target"], as_index=False).agg(
        ca_u=("case_a_present_hateful_union", "sum"),
        cb_u=("case_b_present_nonhateful_union", "sum"),
        ca_e=("case_a_present_hateful_elsherief", "sum"),
        cb_e=("case_b_present_nonhateful_elsherief", "sum"),
    )
    agg["total_u"] = agg["ca_u"] + agg["cb_u"]
    agg["total_e"] = agg["ca_e"] + agg["cb_e"]
    # Require stable pooled count in union (≥ 20) and at least one ElSherief match
    agg = agg[(agg["total_u"] >= 20) & (agg["total_e"] > 0)]
    agg["rate_u"] = agg["ca_u"] / agg["total_u"]
    agg["rate_e"] = agg["ca_e"] / agg["total_e"]
    agg["delta"] = agg["rate_u"] - agg["rate_e"]
    agg = agg.dropna(subset=["delta"])

    agg = agg.sort_values("delta", ascending=True)
    agg["label"] = agg["report_level"].str.title() + ": " + agg["report_target"]

    labels = agg["label"].tolist()
    deltas = agg["delta"].values
    n = len(labels)
    ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars
    y = np.arange(n) * ROW_SPACING

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"] * 0.6, max(2.0, n * 0.20 + 0.6)),  # .tex: width=0.6\linewidth; height formula shrunk (was n*0.32+0.8) to match tighter row spacing/thinner bars below
        constrained_layout=True,
    )
    ax.set_title("")

    colors = [
        STYLE["colors"]["delta_pos"] if d >= 0 else STYLE["colors"]["delta_neg"]
        for d in deltas
    ]
    ax.barh(y, deltas, color=colors, height=0.5)  # was default height=0.8: thinner bars per author request
    ax.axvline(0, color="black", linewidth=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)  # bumped from 6 (was tick_label_fontsize=8 originally): figure narrowed to 0.6x columnwidth (.tex: width=0.6\linewidth)
    ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)  # scaled by ROW_SPACING for consistent bar thickness
    ax.tick_params(axis="x", labelsize=6, labelrotation=90)  # numeric tick labels collided at this panel width
    # Two-line label, further shrunk to fit 0.6x column width (.tex: width=0.6\linewidth)
    ax.set_xlabel(
        "Correct labeling rate Δ\n(union − ElSherief)",
        fontsize=6.5,  # was axis_label_fontsize=10
    )
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    _save(fig, base / "figures_final/fig04a_elsherief_annotation_delta.pdf")


In [ ]:
fig6_elsherief_annotation_delta(data_dir=GFF_DATA_DIR)


### `fig7_elsherief_coverage_delta`

**Output file:** `outputs/figures_final/fig04b_elsherief_coverage_delta.pdf`

**Source TSV(s):** `outputs/stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`

**Pipeline stage:** Stage 4c (`stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`)

**Docstring (verbatim from source):**

```
Horizontal bar chart: Δ presence rate (union − ElSherief) per reporting group.

Source files:
- stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv

Visual encoding:
- All bars green (union always extends or equals ElSherief coverage)
- Sorted ascending (largest gain at top)
- Only groups with a positive coverage gain are shown

Data treatment: the source file has one row per (coding_level, report_level, report_target).
We aggregate across coding levels by summing distinct_dogwhistles_found and
total_glossary_dogwhistles for both corpora, then recompute presence rates from pooled
counts.  Where ElSherief has no glossary entries for a group, its presence rate is 0.
Groups with no union glossary entries are excluded.
```


In [ ]:
def fig7_elsherief_coverage_delta(data_dir: str = "outputs/") -> None:
    """
    Horizontal bar chart: Δ presence rate (union − ElSherief) per reporting group.

    Source files:
    - stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv

    Visual encoding:
    - All bars green (union always extends or equals ElSherief coverage)
    - Sorted ascending (largest gain at top)
    - Only groups with a positive coverage gain are shown

    Data treatment: the source file has one row per (coding_level, report_level, report_target).
    We aggregate across coding levels by summing distinct_dogwhistles_found and
    total_glossary_dogwhistles for both corpora, then recompute presence rates from pooled
    counts.  Where ElSherief has no glossary entries for a group, its presence rate is 0.
    Groups with no union glossary entries are excluded.
    """
    apply_style()
    base = Path(data_dir)
    s4c_p = base / "stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv"
    if not _need([s4c_p], "fig7"):
        return

    df = _read(s4c_p)
    if df.empty:
        return
    required = {
        "report_level",
        "report_target",
        "distinct_dogwhistles_found_union",
        "total_glossary_dogwhistles_union",
        "distinct_dogwhistles_found_elsherief",
        "total_glossary_dogwhistles_elsherief",
    }
    missing_cols = required - set(df.columns)
    if missing_cols:
        print(f"WARNING: fig7: skipping — missing columns {missing_cols}")
        return

    # Pool across coding levels: sum dogwhistle counts, then recompute rates
    agg = df.groupby(["report_level", "report_target"], as_index=False).agg(
        found_u=("distinct_dogwhistles_found_union", "sum"),
        total_u=("total_glossary_dogwhistles_union", "sum"),
        found_e=("distinct_dogwhistles_found_elsherief", "sum"),
        total_e=("total_glossary_dogwhistles_elsherief", "sum"),
    )
    agg = agg[agg["total_u"] > 0]  # must have union glossary entries
    agg["rate_u"] = agg["found_u"] / agg["total_u"]
    # If ElSherief has no glossary entries, its presence rate is 0 by definition
    agg["rate_e"] = agg.apply(
        lambda r: r["found_e"] / r["total_e"] if r["total_e"] > 0 else 0.0, axis=1
    )
    agg["delta"] = agg["rate_u"] - agg["rate_e"]
    agg = agg[agg["delta"] > 0]  # keep only groups where union extends coverage
    agg = agg.sort_values(
        "delta", ascending=True
    )  # ascending → smallest at y=0 (bottom), largest at y=n-1 (top)
    agg["label"] = agg["report_level"].str.title() + ": " + agg["report_target"]

    labels = agg["label"].tolist()
    deltas = agg["delta"].values
    n = len(labels)
    ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars, compresses figure vertically
    y = np.arange(n) * ROW_SPACING

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"] * 0.75, max(2.0, n * 0.16 + 0.6)),  # height formula shrunk (was n*0.22+0.7) to match the tighter row spacing/thinner bars below
        constrained_layout=True,
    )
    ax.set_title("")

    ax.barh(y, deltas, color=STYLE["colors"]["delta_pos"], height=0.5)  # was default height=0.8: compressed per author request
    ax.axvline(0, color="black", linewidth=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)  # bumped from 6 (was tick_label_fontsize=8 originally): figure narrowed to 0.6x columnwidth (.tex: width=0.6\linewidth)
    ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)  # scaled by ROW_SPACING for consistent bar thickness
    ax.tick_params(axis="x", labelsize=6, labelrotation=90)  # numeric tick labels collided at this panel width
    # Two-line label, further shrunk to fit 0.6x column width (.tex: width=0.6\linewidth)
    ax.set_xlabel(
        "Presence rate Δ\n(union − ElSherief)", fontsize=6.5  # was axis_label_fontsize=10
    )
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    _save(fig, base / "figures_final/fig04b_elsherief_coverage_delta.pdf")


In [ ]:
fig7_elsherief_coverage_delta(data_dir=GFF_DATA_DIR)


### `appA_coverage_presence_vs_type_by_level`

**Output file:** `outputs/figures_final/fig06_coverage_presence_vs_type_by_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_coverage_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_coverage_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal stacked bar: presence_rate (dark) and type_coverage gap.

Source files:
- stage4/by_level_group/s4b_coverage_by_level_group.tsv

Visual encoding:
- Dark blue segment = presence_rate (fraction of dogwhistle surface forms found)
- Medium blue gap = type_coverage − presence_rate (found types but not all forms)
- Light gray gap = 1 − type_coverage (entire categories absent from corpus)
- Only non-self-referential rows shown
```


In [ ]:
def appA_coverage_presence_vs_type_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal stacked bar: presence_rate (dark) and type_coverage gap.

    Source files:
    - stage4/by_level_group/s4b_coverage_by_level_group.tsv

    Visual encoding:
    - Dark blue segment = presence_rate (fraction of dogwhistle surface forms found)
    - Medium blue gap = type_coverage − presence_rate (found types but not all forms)
    - Light gray gap = 1 − type_coverage (entire categories absent from corpus)
    - Only non-self-referential rows shown
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_coverage_by_level_group.tsv"
    if not _need([s4b_p], "appA"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["is_self_referential"].astype(bool)].copy()
    df = df.dropna(subset=["presence_rate", "type_coverage"])

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(STYLE["textwidth_in"], 3.5), constrained_layout=True  # was height=5.5: compressed vertically per author request
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = df[df["coding_level"] == level].sort_values(
            "type_coverage", ascending=True
        )

        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["report_target"].tolist()
        n = len(groups)
        y = np.arange(n)

        pr = sub["presence_rate"].values
        tc = sub["type_coverage"].values
        gap_form = np.clip(tc - pr, 0, None)  # form-level gap; 0 when tc < pr
        # Third segment must start where second ends regardless of float quirks
        seg2_end = pr + gap_form  # == max(pr, tc) element-wise
        gap_type = np.clip(1.0 - seg2_end, 0, None)  # categorical absence

        ax.barh(y, pr, color=STYLE["colors"]["presence"], height=0.6, label="Presence rate")  # was default height=0.8: thinner bars per author request
        ax.barh(y, gap_form, left=pr, height=0.6, color="#7EB9DD", label="Type found, form gap")
        ax.barh(
            y, gap_type, left=seg2_end, height=0.6, color="#cccccc", label="Categorical absence"
        )

        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=STYLE["tick_label_fontsize"])
        ax.set_xlim(0, 1.0)
        ax.set_xlabel("Coverage", fontsize=STYLE["axis_label_fontsize"])

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    legend_handles = [
        mpatches.Patch(facecolor=STYLE["colors"]["presence"], label="Presence rate"),
        mpatches.Patch(facecolor="#7EB9DD", label="Type found, form gap"),
        mpatches.Patch(facecolor="#cccccc", label="Categorical absence"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        ncol=3,
        fontsize=STYLE["legend_fontsize"],
        frameon=False,
    )

    _save(fig, base / "figures_final/fig06_coverage_presence_vs_type_by_level.pdf")


In [ ]:
appA_coverage_presence_vs_type_by_level(data_dir=GFF_DATA_DIR)


### `appB_token_frequency_by_level`

**Output file:** `outputs/figures_final/fig07_token_frequency_by_level.pdf`

**Source TSV(s):** `outputs/stage1/s1_coverage_by_level_target.tsv`

**Pipeline stage:** Stage 1 (`stage1/s1_coverage_by_level_target.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal bar: top-10 groups by token frequency per coding level.

Source files:
- stage1/s1_coverage_by_level_target.tsv (token_frequency column)

Visual encoding:
- Bar color matches level color (Okabe-Ito orange/sky-blue/pink)
- Top 10 groups per level by raw token match count
```


In [ ]:
def appB_token_frequency_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar: top-10 groups by token frequency per coding level.

    Source files:
    - stage1/s1_coverage_by_level_target.tsv (token_frequency column)

    Visual encoding:
    - Bar color matches level color (Okabe-Ito orange/sky-blue/pink)
    - Top 10 groups per level by raw token match count
    """
    apply_style()
    base = Path(data_dir)
    s1_p = base / "stage1/s1_coverage_by_level_target.tsv"
    if not _need([s1_p], "appB"):
        return

    s1 = _read(s1_p)
    if s1.empty:
        return

    s1 = s1[~s1["is_self_referential"].astype(bool)]
    # Explicit exclusion (2026-07-03, author request): referential_white_supremacist
    # has is_self_referential=False at L3 in the pipeline data, so the flag alone
    # is insufficient to drop it (same issue documented in annotation_rates_by_level).
    s1 = s1[s1["target"] != "referential_white_supremacist"]

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(5.5, 3.0), constrained_layout=True  # height 5.5->3.0: row spacing was never tightened after the earlier widening, leaving far more vertical room than the text labels need (unlike fig3/fig6, which already got ROW_SPACING)
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = (
            s1[s1["coding_level"] == level]
            .nlargest(10, "token_frequency")
            .sort_values("token_frequency", ascending=True)
        )

        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["target"].tolist()
        freqs = sub["token_frequency"].values
        n = len(groups)
        ROW_SPACING = 0.6  # matches fig3/fig6's row density (author confirmed those look right)
        y = np.arange(n) * ROW_SPACING

        ax.barh(y, freqs, color=STYLE["colors"][level], height=0.4)  # was 0.6: thinner still, matches fig3/fig6's bar-to-gap ratio
        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=8)  # bumped from 6.5: panels are wide now (5.5in/3), room for larger text per author request
        ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6, rotation removed: panels wide enough now
        ax.set_xlabel("Token frequency", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panel wide enough now

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/fig07_token_frequency_by_level.pdf")


In [ ]:
appB_token_frequency_by_level(data_dir=GFF_DATA_DIR)


### `appC_annotation_rates_pooled`

**Output file:** `outputs/figures_final/fig08_annotation_rates_pooled.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_annotation_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_annotation_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Pooled correct vs. failure rate bar chart (all groups, all levels combined).

Source files:
- stage4/by_level_group/s4b_annotation_by_level_group.tsv

Visual encoding:
- Same style as Figure 2 but with data pooled across L2/L3/L4
- Aggregate by summing case_a and case_b counts, then recomputing rates
- Only groups with at least 20 total matches after pooling
```


In [ ]:
def appC_annotation_rates_pooled(data_dir: str = "outputs/") -> None:
    """
    Pooled correct vs. failure rate bar chart (all groups, all levels combined).

    Source files:
    - stage4/by_level_group/s4b_annotation_by_level_group.tsv

    Visual encoding:
    - Same style as Figure 2 but with data pooled across L2/L3/L4
    - Aggregate by summing case_a and case_b counts, then recomputing rates
    - Only groups with at least 20 total matches after pooling
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_annotation_by_level_group.tsv"
    if not _need([s4b_p], "appC"):
        return

    df = _read(s4b_p)
    if df.empty:
        return


    # Pool across levels
    agg = df.groupby("report_target", as_index=False).agg(
        case_a=("case_a_present_hateful", "sum"),
        case_b=("case_b_present_nonhateful", "sum"),
    )
    agg["total"] = agg["case_a"] + agg["case_b"]
    agg = agg[agg["total"] >= 20]
    agg["correct_rate"] = agg["case_a"] / agg["total"]
    agg["failure_rate"] = agg["case_b"] / agg["total"]
    agg = agg.sort_values("failure_rate", ascending=True)

    if agg.empty:
        print("WARNING: appC has no pooled groups with n >= 20")
        return

    groups = agg["report_target"].tolist()
    n = len(groups)
    ROW_SPACING = 0.6  # tighter row spacing (was 1.0 implicit via np.arange): bar height roughly matching text label height per author request
    y = np.arange(n) * ROW_SPACING

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"], max(2.5, n * 0.22 + 0.6)),  # .tex: width=\columnwidth; height formula shrunk (was n*0.38+1.0) to match the tighter row spacing/thinner bars below
        constrained_layout=True,
    )
    ax.set_title("")

    # Stacked (was two dodged bars per category, offset +-BAR_H/2) --
    # correct_rate + failure_rate always sum to 1, so one bar per category
    # with the two rates concatenated communicates the same information more
    # compactly. Matches appA_coverage_presence_vs_type_by_level's stacking
    # convention and annotation_rates_by_level's identical fix, per author request.
    correct = agg["correct_rate"].values
    failure = agg["failure_rate"].values
    ax.barh(y, correct, color=STYLE["colors"]["correct"], height=0.4, label="Correct")  # was default height=0.8: thinner, matches annotation_rates_by_level
    ax.barh(y, failure, left=correct, color=STYLE["colors"]["failure"], height=0.4, label="Failure")

    ax.set_yticks(y)
    ax.set_yticklabels(groups, fontsize=9)  # bumped from tick_label_fontsize=8: bigger text per author request
    ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("Rate", fontsize=STYLE["axis_label_fontsize"])
    ax.axvline(0, color="black", linewidth=0.5)
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    # Legend moved below the plot (was ax-level, lower-right, overlapping the
    # bottom bars once rows were tightened above) -- fig-level legend, outside
    # the axes, matching the convention used elsewhere in this notebook.
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="outside lower center", ncol=2,
        fontsize=STYLE["legend_fontsize"], frameon=False,
    )

    _save(fig, base / "figures_final/fig08_annotation_rates_pooled.pdf")


In [ ]:
appC_annotation_rates_pooled(data_dir=GFF_DATA_DIR)


### `appD_worst_di_and_label_gap_pooled`

**Output file:** `outputs/figures_final/fig_unused_worst_di_and_label_gap_pooled.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Two-panel figure: worst pooled DI ratio (left) and largest label gap (right).

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- Left panel: worst (min) DI ratio across levels per pair — colored pass/fail
- Right panel: largest (max) absolute label gap across levels per pair — all red
- Only stable pairs (unstable_small_n == False) considered
```


In [ ]:
def appD_worst_di_and_label_gap_pooled(data_dir: str = "outputs/") -> None:
    """
    Two-panel figure: worst pooled DI ratio (left) and largest label gap (right).

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - Left panel: worst (min) DI ratio across levels per pair — colored pass/fail
    - Right panel: largest (max) absolute label gap across levels per pair — all red
    - Only stable pairs (unstable_small_n == False) considered
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "appD"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].copy()

    # Pool across levels: take min DI and max label gap per pair
    pooled = df.groupby(["target_a", "target_b"], as_index=False).agg(
        worst_di=("worst_di_ratio", "min"), max_gap=("labeling_rate_gap_abs", "max")
    )
    pooled = pooled.dropna(subset=["worst_di"]).sort_values("worst_di", ascending=True)

    pairs = [f"{a} vs {b}" for a, b in zip(pooled["target_a"], pooled["target_b"])]
    n = len(pairs)
    ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars
    y = np.arange(n) * ROW_SPACING

    fig, (ax_di, ax_gap) = plt.subplots(
        1,
        2,
        figsize=(6.0, max(1.8, n * 0.13 + 0.6)),  # was max(3.0, n*0.16+1.0): floor/slope were still too generous relative to the (already-thinned) bars -- tightened per author request to match fig2/fig3/fig6's row density
        constrained_layout=True,
    )

    # Left: worst DI
    ax_di.set_title("")
    di_colors = [
        STYLE["colors"]["pass"] if v >= 0.80 else STYLE["colors"]["fail"]
        for v in pooled["worst_di"].values
    ]
    ax_di.barh(y, pooled["worst_di"].values, color=di_colors, height=0.4)  # was 0.6 (before that, default 0.8): thinner bars per author request
    ax_di.axvline(0.80, color="red", linewidth=1.2, linestyle="--")
    ax_di.set_yticks(y)
    ax_di.set_yticklabels(pairs, fontsize=5.5)  # was max(5, tick_label_fontsize-1)=7: narrow panel
    ax_di.set_xlim(0, 1.1)
    ax_di.set_xticks([0, 0.5, 1.0])  # added 0.5 marker per author request
    ax_di.tick_params(axis="x", labelsize=6)
    ax_di.set_xlabel("Worst DI ratio\n(pooled)", fontsize=7)  # was axis_label_fontsize=10, one line: collided with ax_gap's label at this panel width
    # left spine restored per author request (2026-07-03) -- was hidden to rely on
    # category labels alone; author wants the axis line visible.
    ax_di.tick_params(left=True, labelleft=True)

    # Right: largest label gap — NaN means no stable label estimate for that pair
    ax_gap.set_title("")
    gap_vals = pooled["max_gap"].values  # NaN for pairs without stable gap
    gap_plot = np.where(np.isnan(gap_vals), 0, gap_vals)  # 0-length bars for NaN pairs
    ax_gap.barh(y, gap_plot, color=STYLE["colors"]["fail"], height=0.4)  # was 0.6 (before that, default 0.8): thinner bars per author request
    ax_gap.set_yticks(y)
    ax_gap.set_yticklabels(pairs, fontsize=5.5)  # was max(5, tick_label_fontsize-1)=7: narrow panel
    # Dynamic xlim: cap at 1.0, add 10% headroom above observed max
    gap_max = float(np.nanmax(gap_vals)) if np.any(~np.isnan(gap_vals)) else 0.5
    ax_gap.set_xlim(0, min(1.0, gap_max * 1.1))
    ax_gap.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
    ax_gap.tick_params(axis="x", labelsize=6)
    ax_gap.set_xlabel(
        "Largest label gap\n(pooled)", fontsize=7  # was axis_label_fontsize=10, one line: collided with ax_di's label at this panel width
    )
    # left spine restored per author request (2026-07-03) -- was hidden to rely on
    # category labels alone; author wants the axis line visible.
    ax_gap.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/fig_unused_worst_di_and_label_gap_pooled.pdf")


In [ ]:
appD_worst_di_and_label_gap_pooled(data_dir=GFF_DATA_DIR)


### `appE_annotation_label_gap_by_level`

**Output file:** `outputs/figures_final/fig11_annotation_label_gap_by_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal bar: absolute correct-labeling-rate gap per pair × level.

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- All bars in red (labeling gaps are always negative findings)
- Sorted descending (largest gap at top)
- Only stable pairs (unstable_small_n == False) with non-null gap
```


In [ ]:
def appE_annotation_label_gap_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar: absolute correct-labeling-rate gap per pair × level.

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - All bars in red (labeling gaps are always negative findings)
    - Sorted descending (largest gap at top)
    - Only stable pairs (unstable_small_n == False) with non-null gap
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "appE"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].dropna(
        subset=["labeling_rate_gap_abs"]
    )

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(6.4, 1.65), constrained_layout=True  # was (6.4, 2.05): shrunk further to match tighter ROW_SPACING below, keeping bar thickness (not just whitespace) constant
    )

    # Max bar count across panels: gives every bar the same physical thickness
    # regardless of panel, so a sparse panel reads as "mostly empty" rather
    # than a single bar stretched to fill the whole axis.
    max_n = max(1, max((df["coding_level"] == lvl).sum() for lvl in LEVELS))

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = df[df["coding_level"] == level].sort_values(
            "labeling_rate_gap_abs", ascending=False
        )

        if sub.empty:
            ax.axis("off")
            continue

        pairs = [f"{a} vs {b}" for a, b in zip(sub["target_a"], sub["target_b"])]
        gaps = sub["labeling_rate_gap_abs"].values
        n = len(pairs)
        ROW_SPACING = 0.36  # was 0.5 (before that, 0.6): gap between bars tightened per author request
        y = np.arange(n) * ROW_SPACING

        ax.barh(y, gaps, color=STYLE["colors"]["fail"], height=0.26)  # was 0.35 (before that, 0.4): further thinned per author request
        ax.set_yticks(y)
        ax.set_yticklabels(pairs, fontsize=5)  # kept at 5, NOT bumped: full "X vs Y" pair strings are long enough that bumping to 6 made labels bleed across panels even with extra wspace; correctness over the cosmetic bump here
        x_max = float(gaps.max()) * 1.15 if len(gaps) else 1.0  # per-panel dynamic range (was fixed [0,1] across all panels) -- author wants each subplot scaled to its own data
        ax.set_xlim(0, x_max)
        ax.set_xticks([0, x_max / 2, x_max])  # midpoint marker computed per panel, not a hardcoded 0.5
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6, rotation removed: panel is wide enough now
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # consistent bar thickness across panels, scaled by ROW_SPACING
        ax.set_xlabel(
            "Label gap\n(|Δ correct rate|)", fontsize=7  # was axis_label_fontsize=10
        )

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/fig11_annotation_label_gap_by_level.pdf")


In [ ]:
appE_annotation_label_gap_by_level(data_dir=GFF_DATA_DIR)


### `appF_cross_level_deltas`

**Output file:** `outputs/figures_final/fig12_cross_level_deltas.pdf`

**Source TSV(s):** `outputs/stage3/s3_cross_level_consistency.tsv`

**Pipeline stage:** Stage 3 (`stage3/s3_cross_level_consistency.tsv`)

**Docstring (verbatim from source):**

```
Two-panel slope chart (stacked vertically): presence rate and
correct-labeling rate plotted directly at each coding level, one line per
target group connecting its values across L2 -> L3 -> L4.

Source files:
- stage3/s3_cross_level_consistency.tsv

Visual encoding:
- Top panel: presence_rate per level. Bottom panel: correct_labeling_rate
  per level. Same y-domain [0, 1] in both, sharing the x-axis.
- One line per target group. Only groups with a non-null value at all
  three levels (L2, L3, L4) in *both* presence_rate and
  correct_labeling_rate are plotted, so every line spans the full axis
  and the group set is identical in both panels.
- Color encodes the group's reporting dimension (race, religion, lgbtq,
  gender, origin, politics); marker shape distinguishes individual groups
  sharing a dimension. The same (color, marker) pair is used for a given
  group in both panels.
- Each line is labeled directly at its rightmost available point instead
  of via a legend; overlapping labels are greedily separated in pixel
  space (see _declutter_labels).

Design rationale: the previous 39-row diverging bar chart (one row per
group x transition) required a separate delta number per group to convey
a within-group change. Plotting the raw rates as connected lines lets the
slope itself communicate the change, and consolidates each group into a
single visual element spanning both panels.
```


In [ ]:
def appF_cross_level_deltas(data_dir: str = "outputs/") -> None:
    """
    Two-panel slope chart (stacked vertically): presence rate and
    correct-labeling rate plotted directly at each coding level, one line per
    target group connecting its values across L2 -> L3 -> L4.

    Source files:
    - stage3/s3_cross_level_consistency.tsv

    Visual encoding:
    - Top panel: presence_rate per level. Bottom panel: correct_labeling_rate
      per level. Same y-domain [0, 1] in both, sharing the x-axis.
    - One line per target group. Only groups with a non-null value at all
      three levels (L2, L3, L4) in *both* presence_rate and
      correct_labeling_rate are plotted, so every line spans the full axis
      and the group set is identical in both panels.
    - Color encodes the group's reporting dimension (race, religion, lgbtq,
      gender, origin, politics); marker shape distinguishes individual groups
      sharing a dimension. The same (color, marker) pair is used for a given
      group in both panels.
    - Each line is labeled directly at its rightmost available point instead
      of via a legend; overlapping labels are greedily separated in pixel
      space (see _declutter_labels).

    Design rationale: the previous 39-row diverging bar chart (one row per
    group x transition) required a separate delta number per group to convey
    a within-group change. Plotting the raw rates as connected lines lets the
    slope itself communicate the change, and consolidates each group into a
    single visual element spanning both panels.
    """
    apply_style()
    base = Path(data_dir)
    s3_p = base / "stage3/s3_cross_level_consistency.tsv"
    if not _need([s3_p], "appF"):
        return

    df = _read(s3_p)
    if df.empty:
        return

    # Validate required columns
    required = {
        "taxonomy_level",
        "target",
        "coding_level_from",
        "coding_level_to",
        "presence_rate_from",
        "presence_rate_to",
    }
    missing = required - set(df.columns)
    if missing:
        print(f"WARNING: appF: skipping — missing columns {missing}")
        return

    # Resolve labeling rate columns (accept either naming variant)
    if {"correct_labeling_rate_from", "correct_labeling_rate_to"} <= set(df.columns):
        lab_from_col, lab_to_col = (
            "correct_labeling_rate_from",
            "correct_labeling_rate_to",
        )
    elif {"labeling_rate_from", "labeling_rate_to"} <= set(df.columns):
        lab_from_col, lab_to_col = "labeling_rate_from", "labeling_rate_to"
    else:
        lab_from_col = lab_to_col = None
        print(
            "WARNING: appF: no labeling rate from/to columns found — "
            "right panel will be empty."
        )

    # Same group-exclusion logic as the prior delta version: pan-category and
    # self-referential targets excluded from pipeline reporting.
    EXCLUDE_TARGETS = {
        "referential_white_supremacist",
        "minority",
        "unknown_minority",
        "other",
    }
    df = df[~df["target"].isin(EXCLUDE_TARGETS)].copy()

    LEVELS = ["L2", "L3", "L4"]

    # Dimension display mapping, consistent with fig1_coverage_heatmap.
    DIM_MAP = {
        "race": "race",
        "religion": "religion",
        "sexuality": "lgbtq",
        "gender": "gender",
        "origin": "origin",
        "politics": "politics",
        "disability": "disability",
    }
    DIM_ORDER = [
        "race",
        "religion",
        "lgbtq",
        "gender",
        "origin",
        "politics",
        "disability",
    ]
    DIM_COLORS = {
        "race": "#0072B2",
        "religion": "#D55E00",
        "lgbtq": "#009E73",
        "gender": "#CC79A7",
        "origin": "#E69F00",
        "politics": "#56B4E9",
        "disability": "#999999",
    }
    MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

    groups: list[tuple[str, str]] = sorted(
        df[["taxonomy_level", "target"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    if not groups:
        print("WARNING: appF: no groups remain after filtering — skipping.")
        return

    # Build per-group level -> rate dicts from the from/to row pairs.
    pres_series: dict[tuple[str, str], dict[str, float]] = {}
    lab_series: dict[tuple[str, str], dict[str, float]] = {}
    for tl, tg in groups:
        sub = df[(df["taxonomy_level"] == tl) & (df["target"] == tg)]
        pres: dict[str, float] = {}
        lab: dict[str, float] = {}
        for _, row in sub.iterrows():
            pres[row["coding_level_from"]] = row["presence_rate_from"]
            pres[row["coding_level_to"]] = row["presence_rate_to"]
            if lab_from_col is not None:
                lab[row["coding_level_from"]] = row[lab_from_col]
                lab[row["coding_level_to"]] = row[lab_to_col]
        pres_series[(tl, tg)] = pres
        lab_series[(tl, tg)] = lab

    # Keep only groups with a complete L2, L3, L4 trajectory in *both* metrics,
    # so every line spans the full x-axis and the same group set appears in
    # both panels (required for cross-panel color/marker tracking).
    def _is_complete(level_dict: dict[str, float]) -> bool:
        return all(
            lvl in level_dict and not pd.isna(level_dict[lvl]) for lvl in LEVELS
        )

    groups = [
        g
        for g in groups
        if _is_complete(pres_series[g]) and _is_complete(lab_series[g])
    ]
    if not groups:
        print("WARNING: appF: no groups have complete L2/L3/L4 data in both "
              "metrics — skipping.")
        return

    # Assign (color, marker): color by dimension, marker cycles within dimension.
    group_style: dict[tuple[str, str], tuple[str, str]] = {}
    for dim in DIM_ORDER:
        dim_groups = [g for g in groups if DIM_MAP.get(g[0]) == dim]
        for i, g in enumerate(dim_groups):
            group_style[g] = (DIM_COLORS[dim], MARKERS[i % len(MARKERS)])

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(STYLE["textwidth_in"], 6.5),  # height bumped: label-crowding fix (see below)
        constrained_layout=True,
    )

    fig.set_constrained_layout_pads(hspace=0.12, wspace=0.0)

    panel_defs = [
        ("Presence rate", pres_series, axes[0]),
        ("Correct-labeling rate", lab_series, axes[1]),
    ]

    # Stretch the inter-level spacing so the lines use more of the panel's
    # horizontal room instead of being compressed into its left half, while
    # the right-hand pad (reserved for end-of-line labels) stays small
    # enough that the longest label string still fits without overflowing.
    LEVEL_SPACING = 2.2
    RIGHT_PAD = 1.3
    x_idx = [i * LEVEL_SPACING for i in range(len(LEVELS))]
    label_fontsize = STYLE["annotation_fontsize"] + 1.5
    summaries: list[str] = []

    for title, series_map, ax in panel_defs:
        ax.set_title("")

        end_points: list[tuple[str, float, float, str]] = []
        for g in groups:
            series = series_map.get(g, {})
            ys = [series.get(lvl, np.nan) for lvl in LEVELS]
            ys = [float(v) if v is not None and not pd.isna(v) else np.nan for v in ys]
            if all(np.isnan(v) for v in ys):
                continue

            color, marker = group_style[g]
            ax.plot(
                x_idx,
                ys,
                color=color,
                marker=marker,
                markersize=4.5,
                linewidth=1.3,
                alpha=0.85,
                clip_on=False,
                zorder=3,
            )

            valid_idx = [i for i, v in enumerate(ys) if not np.isnan(v)]
            x_final = x_idx[valid_idx[-1]]
            y_final = ys[valid_idx[-1]]
            tl, tg = g
            end_points.append((f"{DIM_MAP.get(tl, tl)}:{tg}", x_final, y_final, color))

        ax.set_xlim(-0.15 * LEVEL_SPACING, x_idx[-1] + RIGHT_PAD)
        ax.set_xticks(x_idx)
        ax.set_xticklabels(LEVELS)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel(title, fontsize=STYLE["axis_label_fontsize"])
        ax.tick_params(axis="x", labelsize=STYLE["tick_label_fontsize"])
        ax.tick_params(axis="y", labelsize=STYLE["tick_label_fontsize"])

        _declutter_labels(ax, fig, end_points, label_fontsize)

        summaries.append(f"  {title}: {len(end_points)} groups plotted")

    _save(fig, base / "figures_final/fig12_cross_level_deltas.pdf")

    print("appF summary:")
    for s in summaries:
        print(s)


In [ ]:
appF_cross_level_deltas(data_dir=GFF_DATA_DIR)


### `appG_elsherief_pairwise_di_delta`

**Output file:** `outputs/figures_final/fig13_elsherief_pairwise_di_delta.pdf`

**Source TSV(s):** `outputs/stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv`

**Pipeline stage:** Stage 4c (`stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv`)

**Docstring (verbatim from source):**

```
Scatter plot: Δ worst DI (x) vs Δ label gap (y) for group pairs comparing union to ElSherief.

Source files:
- stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv

Visual encoding:
- Each point = one group pair with stable estimates in both conditions
- Dashed gray lines at x=0 and y=0 divide the four quadrants
- Points labeled with pair names in 7pt font
- Points above y=0 line = worse label gap in union; below = better label gap
```


In [ ]:
def appG_elsherief_pairwise_di_delta(data_dir: str = "outputs/") -> None:
    """
    Scatter plot: Δ worst DI (x) vs Δ label gap (y) for group pairs comparing union to ElSherief.

    Source files:
    - stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv

    Visual encoding:
    - Each point = one group pair with stable estimates in both conditions
    - Dashed gray lines at x=0 and y=0 divide the four quadrants
    - Points labeled with pair names in 7pt font
    - Points above y=0 line = worse label gap in union; below = better label gap
    """
    apply_style()
    base = Path(data_dir)
    s4c_p = base / "stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv"
    if not _need([s4c_p], "appG"):
        return

    df = _read(s4c_p)
    if df.empty:
        return

    df = (
        df[
            ~df["unstable_small_n_union"].astype(bool)
            & ~df["unstable_small_n_elsherief"].astype(bool)
        ]
        .dropna(
            subset=[
                "worst_di_ratio_delta_union_minus_elsherief",
                "label_gap_abs_delta_union_minus_elsherief",
            ]
        )
        .copy()
    )

    if df.empty:
        print("WARNING: appG: no stable pairs in both conditions")
        return

    x = df["worst_di_ratio_delta_union_minus_elsherief"].values
    y_vals = df["label_gap_abs_delta_union_minus_elsherief"].values
    pair_labels = [f"{a}/{b}" for a, b in zip(df["target_a"], df["target_b"])]

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"], STYLE["columnwidth_in"] * 0.6),  # height 1.0x->0.6x columnwidth: shorter y-axis to save vertical space in-column, per author request
        constrained_layout=True,
    )
    ax.set_title("")

    ax.scatter(x, y_vals, s=30, color=STYLE["colors"]["presence"], alpha=0.8, zorder=3)

    # Plain ax.text with fixed (x, y) offsets got two labels overlapping
    # (liberal/republican and LGB/Trans/NB sit at nearly the same y); reuse
    # the pixel-space decluttering helper (already used by appF) instead of
    # a manual per-label offset, so it stays correct if the underlying pairs change.
    end_points = [(lbl, xi, yi, "black") for xi, yi, lbl in zip(x, y_vals, pair_labels)]
    _declutter_labels(ax, fig, end_points, fontsize=8.4, x_pad_px=40.0)  # was default x_pad_px=6.0 (then 14.0, 20.0): marker (s=30) still touched the label at those; 40px clears it, per author request

    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", zorder=1)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", zorder=1)

    # Two-line labels to fit within column width (3.35 in)
    ax.set_xlabel(
        "Δ worst DI\n(union − ElSherief)", fontsize=STYLE["axis_label_fontsize"]
    )
    ax.set_ylabel(
        "Δ label gap abs\n(union − ElSherief)", fontsize=STYLE["axis_label_fontsize"]
    )

    _save(fig, base / "figures_final/fig13_elsherief_pairwise_di_delta.pdf")


In [ ]:
appG_elsherief_pairwise_di_delta(data_dir=GFF_DATA_DIR)


## Pairwise annotation DI figure (`generate_fig2_annotation_di_pairwise.py`)

**Output file:** `outputs/figures_final/fig02_annotation_di_by_pair_level.pdf`,
overwritten unconditionally on every run (see `make_figure` below) --
matching every other figure function's `_save()` behavior. Prior to the
2026-09-09 production promotion this refused to overwrite an existing
file (writing `_v2.pdf` instead); that guard made re-running this
notebook not actually update the canonical file, which defeats the
point of a repeatable production entry point -- removed.

**Source TSVs read (two, explicitly merged by this script itself):**
- `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv` -- **Stage 4b**
- `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` -- **Stage 4a**

**Pipeline stage:** Stage 4 (both 4a and 4b, merged with 4b preferred on key
overlap -- see `load_and_merge` below).

This script's own docstring says it matches "Figure 9
(fig10_worst_di_by_pair_level.pdf)" in style, but its output filename uses a
`fig2_` prefix -- a naming inconsistency between the code comment and the
actual artifact name, left as found.

Reuses `STYLE`, `apply_style`, `_read`, `_save`, `_panel_label` from the
`generate_figures_final.py` utilities section above (imported once, not
duplicated), matching the source file's own
`from generate_figures_final import STYLE, _panel_label, _read, _save, apply_style`.

Hardcodes `DI_THRESHOLD = 0.80` as a local module constant rather than
importing `audit_pipeline.config.DI_THRESHOLD` (numerically identical, 0.8 ==
0.80, but a second, independent definition of the same constant in a third
place -- flagged, not consolidated, since consolidating it would be a logic
change beyond copy-paste).


In [ ]:
DATA_DIR = OUTPUTS_DIR  # absolute; original source hardcoded relative Path("outputs/")
S4B_PATH = DATA_DIR / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
S4A_PATH = DATA_DIR / "stage4/by_group/s4a_pairwise_disparity_by_group.tsv"
OUT_PATH = DATA_DIR / "figures_final/fig02_annotation_di_by_pair_level.pdf"

DI_THRESHOLD = 0.80
KEY_COLS = ["target_a", "target_b", "coding_level"]
LEVELS = ["L2", "L3", "L4"]


In [ ]:
def _filter_stable(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df["unstable_small_n"].astype(bool)].dropna(subset=["annotation_di_ratio"])

def load_and_merge() -> pd.DataFrame:
    s4b_stable = _filter_stable(_read(S4B_PATH)).copy()
    s4a_stable = _filter_stable(_read(S4A_PATH)).copy()
    s4b_stable["source_file"] = "s4b"
    s4a_stable["source_file"] = "s4a"

    s4b_keys = set(map(tuple, s4b_stable[KEY_COLS].values))
    s4a_keys = set(map(tuple, s4a_stable[KEY_COLS].values))
    overlap_keys = s4b_keys & s4a_keys

    print("=" * 80)
    print("OVERLAP CHECK (s4a vs s4b, stable rows only)")
    print("=" * 80)
    if overlap_keys:
        print(f"{len(overlap_keys)} pair x coding_level combination(s) appear in BOTH files. "
              f"Using s4b value (finer granularity) for these; dropping the s4a duplicate.")
        overlap_mask_a = s4a_stable[KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)
        overlap_mask_b = s4b_stable[KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)
        cols = KEY_COLS + ["annotation_di_ratio"]
        print("-- overlapping s4a rows (dropped) --")
        print(s4a_stable[overlap_mask_a][cols].to_string(index=False))
        print("-- overlapping s4b rows (kept) --")
        print(s4b_stable[overlap_mask_b][cols].to_string(index=False))
        s4a_stable = s4a_stable[~overlap_mask_a]
    else:
        print("No overlapping pair x coding_level combinations found.")
    print("=" * 80 + "\n")

    return pd.concat([s4b_stable, s4a_stable], ignore_index=True)

def build_pair_table(merged: pd.DataFrame) -> pd.DataFrame:
    # target_a/target_b already use the paper's canonical casing (e.g. "LGB",
    # "Trans/NB"; lowercase for everything else) — do not force-lowercase here,
    # or acronyms/proper nouns lose their casing.
    out = pd.DataFrame(
        {
            "coding_level": merged["coding_level"],
            "pair_label": [
                f"{a} vs {b}" for a, b in zip(merged["target_a"], merged["target_b"])
            ],
            "annotation_di_ratio": merged["annotation_di_ratio"],
            "passes_4_5_rule": merged["annotation_di_ratio"] >= DI_THRESHOLD,
            "source_file": merged["source_file"],
        }
    )
    level_order = {lvl: i for i, lvl in enumerate(LEVELS)}
    out = out.assign(_lvl=out["coding_level"].map(level_order))
    out = out.sort_values(["_lvl", "annotation_di_ratio"], ascending=[True, True])
    return out.drop(columns="_lvl").reset_index(drop=True)

def compute_level_candidate_counts(raw_s4b: pd.DataFrame, raw_s4a: pd.DataFrame) -> dict[str, dict[str, int]]:
    """Per-level count of all candidate pairs (pre stability-filter) vs. how many
    were excluded as unstable_small_n. Used to explain sparse panels honestly."""
    s4b = raw_s4b.copy()
    s4a = raw_s4a.copy()
    s4b_keys = set(map(tuple, s4b[KEY_COLS].values))
    s4a_unique = s4a[~s4a[KEY_COLS].apply(tuple, axis=1).isin(s4b_keys)]
    raw_merged = pd.concat([s4b, s4a_unique], ignore_index=True)

    counts = {}
    for level in LEVELS:
        sub = raw_merged[raw_merged["coding_level"] == level]
        total = len(sub)
        unstable = int(sub["unstable_small_n"].astype(bool).sum())
        counts[level] = {"total": total, "unstable": unstable, "stable": total - unstable}
    return counts

def make_figure(table: pd.DataFrame, level_counts: dict[str, dict[str, int]]) -> Path:
    apply_style()
    fig, axes = plt.subplots(
        1, 3, figsize=(STYLE["textwidth_in"] * 1.1, STYLE["textwidth_in"] * 0.26), constrained_layout=True  # width *1.0->*1.1: rightmost panel's "Annotation DI ratio" xlabel was getting clipped ("...rati"); wider canvas gives constrained_layout enough room, per author request
    )

    panel_counts: dict[str, int] = {}
    panel_fails: dict[str, list[str]] = {}

    # Max bar count across panels: used to give every bar the same physical
    # thickness regardless of panel, so a sparse panel reads as "mostly empty"
    # rather than "one giant bar" — the resulting empty rows are then labeled.
    max_n = max(
        1, max(len(table[table["coding_level"] == lvl]) for lvl in LEVELS)
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = table[table["coding_level"] == level].sort_values(
            "annotation_di_ratio", ascending=True
        )
        n = len(sub)

        if sub.empty:
            ax.axis("off")
            ax.text(
                0.5,
                0.5,
                "No stable pairs\nat this level",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=STYLE["annotation_fontsize"],
                color=STYLE["colors"]["neutral"],
            )
            _panel_label(ax, level, STYLE["colors"][level])
            panel_counts[level] = 0
            panel_fails[level] = []
            continue

        pairs = sub["pair_label"].tolist()
        di = sub["annotation_di_ratio"].values
        ROW_SPACING = 0.35  # was 0.45: further tightened row spacing per author request
        y = np.arange(n) * ROW_SPACING

        colors = [
            STYLE["colors"]["pass"] if v >= DI_THRESHOLD else STYLE["colors"]["fail"]
            for v in di
        ]

        ax.barh(y, di, color=colors, height=0.22)  # was 0.3 (before that, 0.4, 0.6, 0.8): further thinned per author request
        ax.axvline(DI_THRESHOLD, color="red", linewidth=1.2, linestyle="--")

        ax.set_yticks(y)
        ax.set_yticklabels(pairs, fontsize=8)  # panels are now full textwidth/3 each (was 0.6x, narrower): room for larger labels again
        ax.set_xlim(0.0, 1.0)
        ax.set_xticks([0.0, 0.5, 1.0])  # restored middle tick: panels wide enough now that 3 ticks no longer collide
        ax.tick_params(axis="x", labelsize=7)
        # Fix the y-range to the busiest panel's bar count so bar thickness is
        # consistent across panels; sparse panels get genuine blank space
        # instead of a single bar stretched to fill the whole axis.
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # scaled by ROW_SPACING for consistent bar thickness across panels
        ax.set_xlabel("Annotation DI ratio", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panels wide enough now

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

        # 2026-07-03: the sparse-panel explanatory note (gray text) that used to
        # render here was removed at the author's request -- it overlaid the L4
        # panel's visible bar/label. panel_counts/panel_fails below still feed the
        # printed candidate-count summary at the end of this cell.
        panel_counts[level] = n
        panel_fails[level] = [p for p, v in zip(pairs, di) if v < DI_THRESHOLD]

    # 2026-09-09 production promotion: this used to refuse to overwrite an
    # existing file (writing "_v2" instead) as a one-time safety net during
    # manual, interactive experimentation. Now that this notebook is the
    # standard, repeatable production entry point (re-run via `jupyter
    # nbconvert --execute` whenever upstream data changes), that guard meant
    # every re-run after the first silently left the canonical file stale
    # and spawned a fresh "_v2" duplicate instead of updating it -- the same
    # class of bug this whole promotion was fixing. Overwrite unconditionally,
    # matching every other figure's _save() behavior (see utilities section).
    out_path = OUT_PATH
    _save(fig, out_path)

    print(f"\nOutput file: {out_path}")
    for level in LEVELS:
        print(f"  {level}: {panel_counts[level]} bars")
    for level in LEVELS:
        fails = panel_fails[level]
        print(f"  {level} red (fail) pairs: {fails if fails else 'none'}")

    print("\nStyle parameters matched to Figure 9 (fig10_worst_di_by_pair_level.pdf):")
    print(f"  figsize = ({STYLE['textwidth_in'] * 1.1}, {STYLE['textwidth_in'] * 0.26}) in, dpi = {STYLE['dpi']}")
    print(
        f"  font.size (tick label) = {STYLE['tick_label_fontsize']}, "
        f"axis label fontsize = {STYLE['axis_label_fontsize']}, "
        f"panel label fontsize = {STYLE['panel_label_fontsize']}"
    )
    print(
        f"  pass (blue) color = {STYLE['colors']['pass']}, "
        f"fail (red) color = {STYLE['colors']['fail']}"
    )
    print("  threshold line: color=red, linewidth=1.2, linestyle='--'")
    print(
        f"  panel label colors: L2={STYLE['colors']['L2']}, "
        f"L3={STYLE['colors']['L3']}, L4={STYLE['colors']['L4']}"
    )
    return out_path


In [ ]:
def main() -> None:
    merged = load_and_merge()
    table = build_pair_table(merged)
    level_counts = compute_level_candidate_counts(_read(S4B_PATH), _read(S4A_PATH))

    print("=" * 80)
    print("VERIFICATION TABLE — stable pairwise annotation DI ratios by coding level")
    print("=" * 80)
    print(table.to_string(index=False))
    print("=" * 80 + "\n")

    print("Candidate pair counts by level (context for sparse panels):")
    for level in LEVELS:
        c = level_counts[level]
        print(
            f"  {level}: {c['stable']} stable / {c['total']} candidate "
            f"({c['unstable']} excluded as unstable_small_n)"
        )
    print()

    make_figure(table, level_counts)


In [ ]:
main()


## Notebook scope -- what this covers and what it doesn't

This notebook produces exactly the figures embedded in (or generated for,
in `appD`'s case, but not embedded in) the paper: `fig01`-`fig12` above plus
`fig_unused_worst_di_and_label_gap_pooled` (14 cells total, matching the
"Camera-ready figures" and "Pairwise annotation DI figure" sections above).

It intentionally does **not** cover `stage5_figures.py`'s PNG outputs, the
deprecated `auditing/` pipeline's PNG outputs, or the one-time cross-stage
consistency check that validated both against `audit_pipeline`'s stage1-4 --
none of those produce a paper figure. That content was split out to
`deprecated/legacy_figures_consolidated.ipynb` on 2026-09-10 (see this
notebook's title cell), and writes under `deprecated/legacy_outputs/` rather
than `outputs/`.

For the full provenance/inventory writeup covering *all four* original
figure-generation code paths (including the two now-archived ones), see
`deprecated/legacy_figures_consolidated.ipynb`'s "Step 4 -- Final summary"
cell, kept there as a historical record rather than duplicated here.

# Style Pass Summary (2026-07-03)

**Scope:** per author direction, careful render-and-verify styling was applied
to the 14 figures actually embedded in the paper's `.tex`
(`generate_figures_final.py`'s 13 + `generate_fig2_annotation_di_pairwise.py`'s
1). The 16 `stage5_figures.py` PNGs and 8 deprecated `auditing/` PNGs got the
same mechanical palette-harmonization and proportional resize, but were not
individually render-verified, since none of them appear in the paper.

**fig2/fig8 collision (Step 1):** `fig2_fig01_annotation_rates_by_level.pdf` renamed
to `fig01_annotation_rates_by_level.pdf` (position-independent, per author choice)
across the notebook, `generate_figures_final.py`, and `acl_latex (5).tex`.
Confirmed position: **Figure 1** in the main body (first `\begin{figure}`
before `\appendix` in `acl_latex (5).tex`) -- not "Figure 8" as originally
believed; that discrepancy was reported, not silently resolved.
`fig02_annotation_di_by_pair_level.pdf` (from `generate_fig2_annotation_di_pairwise.py`)
is confirmed as the real Figure 2, unchanged.

**Deprecation (Step 2):** `stage5_figures.py`, `generate_figures_final.py`,
and `generate_fig2_annotation_di_pairwise.py` each got a module-level
deprecation banner and an abort-by-default `__main__` guard
(`--i-know-this-is-deprecated` flag or `I_KNOW_THIS_IS_DEPRECATED=1` env var
to opt in). All three tested: direct execution aborts with no writes and
exit code 1 without the opt-in; both opt-in mechanisms verified to bypass
correctly. Imports are unaffected.

**Sandbox (Step 3):** the prior pass's monkeypatch-after-definition ordering
hazard is fixed structurally -- `OUTPUTS_DIR` itself is redirected to
`outputs/_style_pass_sandbox/` (git-ignored) in the first executable cell,
before any figure code is defined. A canary cell proves the redirect is live
before any real plotting runs. `SANDBOX_MODE = True` for this run; flipping
it to `False` is the one-line change described in the promotion note above.

**Style changes applied (Step 4):** all changes are figsize, color hex,
fontsize, tick locator/rotation, or padding-constant edits -- see the diff
walkthrough in the chat for the complete list. Highlights:
- Corrected figsize width to match confirmed `.tex` `\includegraphics` widths
  for every embedded figure that was mismatched (most were sized for full
  double-column width but embedded at single-column or partial width).
- Fixed a real regression the width correction exposed: 3-panel figures at
  `\columnwidth` had y-axis category labels bleeding across panels and
  x-axis tick labels colliding. Fixed via smaller tick fonts, fewer/rotated
  ticks -- verified by rendering at final size, not by eyeballing the
  larger original.
- Harmonized the `stage5_figures.py` palette (`_BLUE`/`_GREEN`/`_ORANGE`/
  `_RED`/`_PURPLE`/`_TEAL`, `_CL_COLOR` L2-L4) to exactly match
  `generate_figures_final.py`'s `STYLE["colors"]` hex values, at their single
  definition point.
- Bumped `appF_cross_level_deltas`'s label-decluttering gap and figure
  height for the known line-end-label crowding issue.

**Content-equivalence (Step 5):** zero changed lines across all 19 touched
cells reference any data/threshold/filter/sort keyword (full diff reviewed
line-by-line). Runtime data-extraction spot-check on the two highest-stakes
figures (`annotation_rates_by_level` = Figure 1, `fig2_annotation_di_by_pair_level`
= Figure 2) confirms identical bar widths/heights and text content before vs.
after.

**Known bugs / open questions:** the `_top_n` sort-direction bug noted
in the prior pass has since been fixed (see "Known bugs fixed" in the
Step 4 summary above); the Stage 3 vs. 4a vs. 4b canonical-pairwise-source
question from the prior pass remains open and is unchanged by this pass.

**Production promotion:** deferred. `SANDBOX_MODE` is still `True` in this
saved notebook; no real `outputs/stage5/**`, `outputs/figures_final/**`, or
`outputs/audit_visualizations/**` path was written to during this pass.


## Addendum: second round of fixes from author visual review (2026-07-03)

After the first sandbox render, the author reviewed the actual PDFs and found
four more issues that only showed up at final rendered size:

1. **`fig5_worst_di_by_pair_level` / `appE_annotation_label_gap_by_level`:**
   sparse panels (e.g. L4, often 1 pair) showed a single bar stretched to
   fill the whole panel height instead of matching the other panels' bar
   thickness. Fixed by porting the `max_n`-based fixed-`ylim` technique
   already used in `generate_fig2_annotation_di_pairwise.py`'s `make_figure`
   to both.
2. **`fig2_annotation_di_by_pair_level`:** the gray sparse-panel explanatory
   note (`STYLE["colors"]["neutral"]`) overlaid the L4 panel's visible bar
   and label. Removed entirely per author request.
3. **`fig1_coverage_heatmap`:** three vertical white/light gridlines were
   cutting through the L2/L3/L4 columns. Root cause: `stage5_figures.py`'s
   utilities cell sets `axes.grid`/`axes.grid.axis`/`axes.facecolor` globally
   via `mpl.rcParams.update()`, and `generate_figures_final.py`'s
   `apply_style()` never reset them back -- every gff figure was silently
   inheriting stage5's grid lines and off-white background from shared
   kernel-global rcParams state. Fixed by having `apply_style()` explicitly
   reset `axes.grid: False` / `axes.facecolor: white`, so it's insulated
   regardless of what ran before it in the same kernel.
4. **Label text size bumped** across the bar-plot figures that had been
   shrunk to fix panel-width crowding (`annotation_rates_by_level`, `fig3`,
   `appB`, `fig6`, `fig7`, `fig2_annotation_di_by_pair_level`). Attempted the
   same bump on `fig5`/`appE`, but their labels are full "X vs Y" pair
   strings (longer than the single-word category labels elsewhere) and the
   bump reintroduced cross-panel text bleed even with extra `wspace` --
   reverted those two back to their original (smaller) size rather than ship
   a regression for a cosmetic bump.

Re-executed top-to-bottom after these fixes: 0 errors, canary passed, real
production paths still untouched, outputs kept in this saved file.


## Addendum: third round of fixes from author visual review (2026-07-03, cont.)

1. **`fig2_annotation_di_by_pair_level`**: resized to a 3:2 aspect ratio at
   full `\textwidth` (was 0.6x width, 4.18in), so it spans the top of the
   page in its existing `figure*` environment. `acl_latex (5).tex` updated
   to `\includegraphics[width=\textwidth]`. Bars thinned (`height=0.6`),
   x-ticks restored to `[0, 0.5, 1.0]` and labels back to standard size now
   that panels have more room; left spine restored.
2. **Left y-axis spine restored** on `fig2_annotation_di_by_pair_level`,
   `fig3_case_ab_counts_by_level`, `appB_token_frequency_by_level`,
   `appD_worst_di_and_label_gap_pooled`, `appE_annotation_label_gap_by_level`
   -- these had `ax.spines["left"].set_visible(False)` /
   `tick_params(left=False)` from the original source (pre-dating this style
   pass); author wants the axis line visible. `fig5_worst_di_by_pair_level`,
   `fig6`/`fig7` (ElSherief), and `appC` were not in the author's list and
   were left with the spine hidden, matching original behavior -- flagged in
   chat in case the author wants those changed too for consistency.
3. **Aspect ratio brought closer to 1:1** for `fig5_worst_di_by_pair_level`
   (was 3.35x6.5in, now 3.35x4.0in, aspect 0.84), `fig3_case_ab_counts_by_level`
   (3.35x4.0in, 0.84), and `appD_worst_di_and_label_gap_pooled` (3.35x~3.0in,
   1.11) -- all were unnecessarily tall, making differently-sized bars look
   similar. Fixed via thinner bars (`height=0.6`, was default 0.8) rather
   than just shrinking the figure and letting bars visually merge.
4. **Text-cutoff check**: `_save`'s `bbox_inches="tight"` structurally
   prevents label clipping (the saved page always expands to include all
   rendered text) -- confirmed via direct PDF page-bbox inspection
   (`fitz`/PyMuPDF) rather than assuming. `appD`'s two-panel x-axis labels
   ("Worst DI ratio (pooled)" / "Largest label gap (pooled)") were
   overlapping each other at the narrow panel width (not clipped, but
   colliding) -- fixed by wrapping each onto two lines and shrinking to
   match its sibling figures.

Re-executed top-to-bottom after these fixes: 0 errors, real production paths
still untouched, outputs kept in this saved file.


## Addendum: fourth round of fixes from author visual review (2026-07-03, cont.)

1. **`appB_token_frequency_by_level`, `appD_worst_di_and_label_gap_pooled`**:
   author reported these still much too vertically long. Fixed by
   *widening* rather than shortening -- since `.tex` scales embedded width
   to a fixed target (`\columnwidth` for `appB`; `appD` is not currently
   embedded at all, commented out), a wider source canvas at the same data
   range makes the eventual on-page aspect ratio squarer without touching
   any font/bar-thickness settings. `appB`: (3.35, 5.5) -> (5.5, 5.5),
   aspect now exactly 1.00. `appD`: (3.35, ~3.0) -> (6.0, ~3.0), aspect 1.98
   (wide, per author's explicit "increasing horizontal scale" request).
   Bonus: `appB`'s previously near-invisible low-count bars are now clearly
   visible bars at this wider scale, and its x-tick labels no longer need
   90-degree rotation.
2. **`fig5_worst_di_by_pair_level`**: left spine was still hidden (this
   figure wasn't in the original 5-figure spine-restoration list from the
   previous round) -- restored per this round's explicit request.
3. **Further vertical compression** on `fig5_worst_di_by_pair_level`,
   `appE_annotation_label_gap_by_level`, `fig2_annotation_di_by_pair_level`:
   added an explicit `ROW_SPACING = 0.6` multiplier on the y-positions (was
   1.0 implicit via `np.arange`), decreasing the gap between bars, and
   thinned bars further (`height=0.4`, was `0.6`). Figure heights reduced
   accordingly. `fig2_annotation_di_by_pair_level`'s aspect ratio drifted
   from the previous round's 3:2 target to ~2.2:1 as a direct result --
   explicitly accepted by the author ("aspect ratio is not as important as
   the space being used effectively").

Re-executed top-to-bottom after these fixes: 0 errors, real production
paths still untouched, outputs kept in this saved file. Final aspect ratios
(w:h, measured from the actual saved PDF page box via PyMuPDF): `appB`=1.00,
`appD`=1.98, `fig5`=1.11, `appE`=0.96, `fig2_annotation_di_by_pair_level`=2.19.


## Addendum: fifth round -- batch feedback on generate_figures_final.py outputs (2026-07-03, cont.)

Two naming corrections made before applying this round's feedback (both
confirmed from actual code/renders, not guessed):
- "appB (appB_annotation_case_ab_counts)" does not exist; the described
  content (an unfiltered `referential_white_supremacist` bar) matches the
  real `appB_token_frequency_by_level` -- applied there.
- "fig3 (fig3_cross_level_deltas)" does not exist either; the described
  content ("Match count" label, rightmost-panel cutoff) matches the real
  `fig3_case_ab_counts_by_level` -- applied there. (The actual cross-level
  figure is `appF_cross_level_deltas`, untouched by this round.)

Changes applied, all verified by rendering + PDF page-bbox measurement:

1. `annotation_rates_by_level`: widened to (6.0, 6.0), aspect 1.00. Restored
   `[0, 0.5, 1.0]` x-ticks.
2. `appA_coverage_presence_vs_type_by_level`: left spine restored, height
   5.5->3.5, bars `height=0.6`.
3. `appB_token_frequency_by_level`: bars `height=0.6`, y-tick fontsize
   6.5->8, x-tick fontsize 6->7 (rotation removed, panels wide enough),
   `referential_white_supremacist` dropped (same fix already applied
   elsewhere in this notebook; `is_self_referential` alone doesn't catch it).
4. `appD_worst_di_and_label_gap_pooled`: `ROW_SPACING=0.6`, bars
   `height=0.4` (was 0.6), `[0, 0.5, 1.0]` x-ticks added to the left (DI)
   panel.
5. `appE_annotation_label_gap_by_level`: widened to (6.0, 3.0), fixed
   `set_xlim(0, 1.0)` + `[0, 0.5, 1.0]` ticks (was auto-scaled per panel),
   `ROW_SPACING` 0.6->0.5, bars `height` 0.4->0.35.
6. `fig2_annotation_di_by_pair_level`: `ROW_SPACING` 0.6->0.45, bars
   `height` 0.4->0.3, figure height further reduced (aspect now ~2.9:1,
   drifted further from the 3:2 target per author's explicit go-ahead).
7. `fig3_case_ab_counts_by_level`: widened to (8.0, 4.0), aspect ~2:1 (fixes
   the rightmost panel's "Match count" label getting clipped). `ROW_SPACING
   =0.6`, bars `height` 0.6->0.4. **Fixed a bug introduced by this change**:
   the truncated-bar count annotation (`ax.text` showing the true count next
   to bars clipped by the x-axis) was using the raw enumerate index as its
   y-position, which desynced from the bars once `y` became
   `ROW_SPACING`-scaled -- switched to `zip(y, totals)`.
8. `fig5_worst_di_by_pair_level`: widened to (6.0, 3.0), aspect ~2:1 (fixes
   the rightmost panel's "Worst DI ratio" label getting clipped).
   `[0, 0.5, 1.0]` x-ticks restored. `ROW_SPACING` 0.6->0.45, bars `height`
   0.4->0.3.
9. `fig6_elsherief_annotation_delta`: `ROW_SPACING=0.6` (new), bars
   `height=0.5` (was default 0.8), height formula shrunk to match.
10. `fig7_elsherief_coverage_delta`: bars `height=0.5` (was default 0.8),
    width `0.6x -> 0.75x` columnwidth per author request so bar-length
    differences read more clearly (e.g. Race:white vs Race:latinx).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file. Final aspect ratios (w:h, measured from
actual PDF page box via PyMuPDF): `annotation_rates_by_level`=1.00,
`appA`=1.98, `appB`=1.00, `appD`=1.98, `appE`=1.98,
`fig2_annotation_di_by_pair_level`=2.88, `fig3`=1.99, `fig5`=1.98.


## Addendum: sixth round -- match row density to fig2/fig3/fig6 (2026-07-03, cont.)

Author confirmed `fig2_annotation_di_by_pair_level`, `fig3_case_ab_counts_by_level`,
and `fig6_elsherief_annotation_delta` have the right row density (bar +
padding roughly matching the text label's own height) and asked for
`appB_token_frequency_by_level`, `fig5_worst_di_by_pair_level`, and
`appE_annotation_label_gap_by_level` to match. Root cause: those three had
their bar `height`/`ROW_SPACING` tightened in earlier rounds but their
overall figure *height* was never reduced to match -- leaving unused
vertical room the earlier bar-thinning alone couldn't close.

- `appB_token_frequency_by_level`: had no `ROW_SPACING` at all (still plain
  `np.arange`) despite `fig3`/`fig6` already using one -- added
  `ROW_SPACING=0.6`, bars `height` 0.6->0.4 (matches `fig3`'s ratio), figure
  height 5.5->3.0.
- `fig5_worst_di_by_pair_level`: `ROW_SPACING`/bar-height ratio already
  matched `fig2`'s; figure height alone was too generous -- 3.0->2.4.
- `appE_annotation_label_gap_by_level`: same issue, figure height 3.0->2.4.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: seventh round -- fig5/appD tightening, appE dynamic axes, fig1 aspect (2026-07-03, cont.)

1. `fig5_worst_di_by_pair_level`: figure height 2.4->2.1in, further toward
   fig2/fig3/fig6's row density.
2. `appD_worst_di_and_label_gap_pooled`: height formula floor/slope
   `max(3.0, n*0.16+1.0)` -> `max(1.8, n*0.13+0.6)` -- this had not yet had
   its figure-height examined the way appB/fig5/appE were in the previous
   round, and the floor was binding well above what its ~6 rows need.
3. `appE_annotation_label_gap_by_level`: reverted the fixed `[0,1]` x-axis
   (added last round for a hardcoded 0.5 marker) back to a per-panel
   dynamic range -- each subplot now scales to `gaps.max() * 1.15` with a
   midpoint tick computed from that panel's own range, not a literal 0.5.
   Known minor limitation: the L4 panel here has only one very small-valued
   bar, so its computed midpoint rounds to the same 2-decimal display as
   zero at default formatting -- flagged, not fixed unless it matters to
   the author.
4. `fig1_coverage_heatmap`: width was a fixed `STYLE["textwidth_in"] * 0.67`
   independent of the content-driven height, giving an extreme ~0.35:1
   final aspect for a tall single-page appendix figure. Changed to
   `fig_h * 0.85` (empirically calibrated so the final *saved* PDF -- after
   `bbox_inches="tight"` trims unused margin at the new width -- lands at
   2:3; a naive `fig_h * 2/3` nominal undershoots to ~0.53:1 once trimmed).
   Measured final aspect: 0.669 (target 0.667). `.tex` embed width
   (`0.67\textwidth`) unchanged -- the wider source makes the final printed
   height shrink from ~13in to a reasonable single-page ~7in instead.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: eighth round -- annotation_rates_by_level stacked bars (2026-07-04)

`annotation_rates_by_level` (the paper's actual Figure 1) changed from two
dodged bars per category (`Correct` and `Failure`, offset +-`BAR_H/2`) to a
single stacked bar per category (`Correct` from 0, `Failure` concatenated
via `left=correct`), matching `appA_coverage_presence_vs_type_by_level`'s
stacking convention per author request. `correct_labeling_rate` and
`failure_rate` always sum to 1, so this communicates the same information
in half the vertical space per category. `BAR_H` (only used by the removed
dodged-bar calls) removed as dead code.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: ninth round -- consistent bar thickness in annotation_rates_by_level (2026-07-04)

`annotation_rates_by_level` never had the `max_n`-based fixed-`ylim`
technique (already used in `fig5`/`appE`/`fig2_annotation_di_by_pair_level`)
applied to it, so its sparse L4 panel (4 stable groups vs. L2/L3's 12-14)
auto-scaled its axis to fit just those 4 bars, making them visually much
thicker than L2/L3's. Added `max_n = max(1, max((s2["coding_level"] == lvl).sum()
for lvl in LEVELS))` and `ax.set_ylim(-0.5, max_n - 0.5)` per panel, matching
the convention already used elsewhere. Sparse panels now show genuine blank
space rather than stretched bars.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: tenth round -- bar height/text size in annotation_rates_by_level (2026-07-04, cont.)

Added `ROW_SPACING=0.6` and explicit bar `height=0.4` (was default 0.8),
matching `fig3`/`fig6`'s established row density. Y-tick label fontsize
bumped 7->8. Final aspect stayed close to 1:1 (6.08 x 6.06in).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: eleventh round -- appC_annotation_rates_pooled matches annotation_rates_by_level (2026-07-04, cont.)

Applied the same set of changes just made to `annotation_rates_by_level`
(its unpooled sibling, same underlying Correct/Failure rate semantics):
- Stacked bars (was two dodged bars offset +-`BAR_H/2`) -- `correct_rate`
  and `failure_rate` always sum to 1, matches `appA`'s stacking convention.
- `ROW_SPACING=0.6`, bar `height=0.4` (was default 0.8), figure height
  formula shrunk to match (bar height roughly matching text label height).
- Y-tick label fontsize 8->9 (bigger text).
- **Found and fixed along the way**: the axes-level legend
  (`loc="lower right"`, inside the plot) started overlapping the bottom two
  bars once rows were tightened -- moved to a fig-level legend below the
  plot (`loc="outside lower center"`), matching the convention used
  throughout this notebook (and the same class of fix just applied to
  `fpr_iaa_analysis.ipynb`'s Figure 4).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: twelfth round -- fig7_elsherief_coverage_delta row spacing (2026-07-04, cont.)

`fig7` never had `ROW_SPACING` applied (unlike its sibling `fig6`, which
already used `ROW_SPACING=0.6`) despite both being ElSherief delta bar
charts with the same structure -- added `ROW_SPACING=0.6`, mirroring `fig6`,
and shrunk the figure-height formula to match (was `n*0.22+0.7`, now
`n*0.16+0.6`).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: thirteenth round -- fig2_annotation_di_by_pair_level further compression (2026-07-04, cont.)

Further tightened: `ROW_SPACING` 0.45->0.35, bar `height` 0.3->0.22, figure
height `STYLE["textwidth_in"] * 0.34 -> * 0.26`. Verified via render: bars
remain distinguishable with visible gaps, no text cutoff.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: production promotion (2026-09-09)

**Trigger:** an independent check (rendering `WOAH26_BDW.pdf`'s embedded
Figure 2 and comparing it pixel-by-pixel against this notebook's saved
output vs. against `outputs/figures_final/fig02_annotation_di_by_pair_level.pdf`)
found that the *real* production path still held the pre-style-pass output
from `generate_fig2_annotation_di_pairwise.py` -- a stray gray annotation
box on the L4 panel, wrong aspect ratio, hidden left spine -- none of which
match the submitted paper. This notebook's own saved cell output matched
the paper exactly. Root cause: `SANDBOX_MODE = True` was never flipped;
"Step 7 -- promote to production," referenced by name in the old Setup
section, was never actually written as a cell.

**Verification performed before promoting:** every one of the 14 figures
`acl_latex.tex` actually embeds (`generate_figures_final.py`'s 13 +
`generate_fig2_annotation_di_pairwise.py`'s 1; `appD_worst_di_and_label_gap_pooled`
is generated but confirmed commented out in the `.tex` and correctly excluded)
was rendered from a fresh top-to-bottom sandboxed run of this notebook and
compared page-by-page against the corresponding page of `WOAH26_BDW.pdf`
(rendered via PyMuPDF at 150dpi). All 14 matched exactly -- same values,
same colors, same category ordering, same annotations (or correct absence
thereof, e.g. `fig2`'s removed L4 gray box). None required further edits.

**Changes made:**
1. Removed `SANDBOX_MODE` / `SANDBOX_DIR` and the two canary-check cells
   from the Setup section. `OUTPUTS_DIR` now unconditionally resolves to the
   real `outputs/` tree -- there is no flag left that can silently redirect
   writes away from production again.
2. Cleared the stale `outputs/figures_final/fig02_annotation_di_by_pair_level.pdf`
   and `..._v2.pdf` left by the last direct run of
   `generate_fig2_annotation_di_pairwise.py`, so this notebook's `make_figure`
   (which refuses to overwrite an existing file and would otherwise have
   written a fresh `_v2`/`_v3` instead of the canonical filename) writes to
   the correct path.
3. Re-executed top-to-bottom for real: all 15 candidate figures (14 embedded
   + `appD`) written to `outputs/figures_final/`.
4. `generate_figures_final.py` and `generate_fig2_annotation_di_pairwise.py`
   (repo root) archived to `deprecated/` -- this notebook is now the sole
   figure-generation source; see `deprecated/README.md`.

**Regenerating the figures going forward:**

```bash
jupyter nbconvert --to notebook --execute --inplace \
  audit_pipeline/notebooks/figures_consolidated.ipynb
```

Requires `outputs/stage1`-`outputs/stage4` and `outputs/unioned_data` to
already exist (run `scripts/run_audit_pipeline.sh` first). Writes directly
to `outputs/figures_final/`, `outputs/stage5/`, and
`outputs/audit_visualizations/` -- no flag to set, no sandbox to promote.